# Amazon ML Challenge 2026 — Entity Resolution · Lightning AI Studio · NVIDIA H200 (Jupyter)

**Machine:** Lightning Studio → *Change machine* → **H200** (1x or 8x). Open the **Jupyter** plugin, upload this notebook, Run All.
Everything (data, checkpoints, outputs) lives in `/teamspace/studios/this_studio`, so it survives a machine switch.

How the H200 is used (all auto-detected in the Config cell):
* GPU: SVD projection (torch, TF32), exact top-K with FAISS sharded over every visible H200 (torch top-K fallback if the
  faiss wheel lacks Hopper kernels), XGBoost `device=cuda` with up to 2 GPUs training the two stage-A models in parallel.
* RAM: the training-row cap scales with machine RAM (up to 80M pairs), bigger processing blocks.
* CPU: every core used for normalisation / text hashing (forkserver workers) and numba feature kernels.

**Cost tip:** stages 1 and 3 are CPU-bound; an H200 does not speed them up. Run stage 1 on a cheap CPU Studio, then
switch to the H200 for stages 2–5 (checkpoints make the switch seamless).

| Stage | FULL, 1x H200 machine (estimate) |
|---|---|
| 0 Download + unzip data | ~5 min |
| 1 Normalise | ~5–12 min (depends on vCPUs) |
| 2 Blocking (forward + reverse + 2-hop search, keys) | ~45–70 min |
| 3 Features (train + test, ~2x more pairs than v4) | ~60–90 min |
| 4 Train + tune | ~20–30 min |
| 5 Predict + write | ~30–45 min |
| 6 Cross-encoder (fine-tune + re-score) | ~30–50 min |

Estimates ±50%; each stage logs its own timings.

In [ ]:
# ---- Stage 0: environment -----------------------------------------------------------------
import os, sys, subprocess, shutil
def pip(*p): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *p], check=False)
for mod, pkg in [('faiss', 'faiss-gpu-cu12'), ('rapidfuzz', 'rapidfuzz'), ('numba', 'numba'), ('xgboost', 'xgboost'), ('transformers', 'transformers')]:
    try: __import__(mod)
    except ImportError: pip(pkg)
import numpy as np
for mod in ['faiss', 'rapidfuzz', 'numba', 'xgboost', 'torch', 'transformers']:
    try:
        m = __import__(mod); extra = ''
        if mod == 'faiss': extra = f'GPUs={m.get_num_gpus()}'
        if mod == 'torch': extra = f'GPUs={m.cuda.device_count()}'
        print(f'{mod:10s} {getattr(m, "__version__", "")} {extra}')
    except Exception as e:
        print(f'{mod:10s} MISSING ({e.__class__.__name__}) -> CPU fallback')
mem = [l for l in open('/proc/meminfo') if l.startswith(('MemTotal', 'MemAvailable'))]
print('CPUs', os.cpu_count(), '|', ' '.join(x.split()[1] for x in mem), 'kB')
ROOT = '/teamspace/studios/this_studio' if os.path.isdir('/teamspace/studios/this_studio') else os.getcwd()
print('ROOT', ROOT, 'free %.0f GB' % (shutil.disk_usage(ROOT).free / 1e9), '(need ~100 GB for the FULL run)')

In [ ]:
SRC = f'{ROOT}/er_v4'; os.makedirs(SRC, exist_ok=True)

In [ ]:
%%writefile {SRC}/normalize.py
"""
Text normalisation for business names and addresses.

Design goals
------------
* Country-agnostic: nothing here branches on the `country` label, so France (unseen in
  training) goes through exactly the same code path as US / India.
* Script-agnostic: every Indic script is transliterated to Latin with one table, using the
  fact that the Unicode Indic blocks (Devanagari, Bengali, Gurmukhi, Gujarati, Oriya, Tamil,
  Telugu, Kannada, Malayalam) share the same 128-code-point layout (ISCII heritage).
* Produces several "views" of each string; later stages compare views, not raw text.
"""
import re
import unicodedata

# --------------------------------------------------------------------------------------
# 1. Indic -> Latin transliteration (single table, offset-based)
# --------------------------------------------------------------------------------------
_INDIC_BLOCKS = [0x0900, 0x0980, 0x0A00, 0x0A80, 0x0B00, 0x0B80, 0x0C00, 0x0C80, 0x0D00]
_CONS = {  # offset -> consonant (without inherent vowel)
    0x15: 'k', 0x16: 'kh', 0x17: 'g', 0x18: 'gh', 0x19: 'n', 0x1A: 'ch', 0x1B: 'chh', 0x1C: 'j',
    0x1D: 'jh', 0x1E: 'n', 0x1F: 't', 0x20: 'th', 0x21: 'd', 0x22: 'dh', 0x23: 'n', 0x24: 't',
    0x25: 'th', 0x26: 'd', 0x27: 'dh', 0x28: 'n', 0x29: 'n', 0x2A: 'p', 0x2B: 'ph', 0x2C: 'b',
    0x2D: 'bh', 0x2E: 'm', 0x2F: 'y', 0x30: 'r', 0x31: 'r', 0x32: 'l', 0x33: 'l', 0x34: 'l',
    0x35: 'v', 0x36: 'sh', 0x37: 'sh', 0x38: 's', 0x39: 'h',
    0x58: 'q', 0x59: 'kh', 0x5A: 'g', 0x5B: 'z', 0x5C: 'd', 0x5D: 'rh', 0x5E: 'f', 0x5F: 'y',
}
_VOWEL = {  # independent vowels
    0x05: 'a', 0x06: 'aa', 0x07: 'i', 0x08: 'ii', 0x09: 'u', 0x0A: 'uu', 0x0B: 'ri', 0x0C: 'li',
    0x0D: 'e', 0x0E: 'e', 0x0F: 'e', 0x10: 'ai', 0x11: 'o', 0x12: 'o', 0x13: 'o', 0x14: 'au',
    0x60: 'ri', 0x61: 'li',
}
_MATRA = {  # dependent vowel signs
    0x3E: 'aa', 0x3F: 'i', 0x40: 'ii', 0x41: 'u', 0x42: 'uu', 0x43: 'ri', 0x44: 'ri', 0x45: 'e',
    0x46: 'e', 0x47: 'e', 0x48: 'ai', 0x49: 'o', 0x4A: 'o', 0x4B: 'o', 0x4C: 'au', 0x62: 'li',
    0x63: 'li',
}
_SIGN = {0x01: 'n', 0x02: 'n', 0x03: 'h', 0x50: 'om', 0x70: 'n'}
# Malayalam chillu letters (consonant with no inherent vowel) and au length mark
_CHILLU = {0x7A: 'n', 0x7B: 'n', 0x7C: 'r', 0x7D: 'l', 0x7E: 'l', 0x7F: 'k', 0x54: 'm', 0x55: 'y', 0x56: 'l'}
_SKIP = {0x57, 0x4E, 0x00, 0x3D, 0x64, 0x65}
_VIRAMA, _NUKTA = 0x4D, 0x3C


def _indic_offset(ch):
    cp = ord(ch)
    for base in _INDIC_BLOCKS:
        if base <= cp < base + 0x80:
            return cp - base
    return None


def transliterate(text: str) -> str:
    """Transliterate any Indic-script characters to Latin; leave everything else alone."""
    if not text or all(ord(c) < 0x0900 for c in text):
        return text
    # Malayalam conjuncts whose reading differs from the component letters
    text = text.replace('\u0D31\u0D4D\u0D31', '\u0D1F\u0D4D\u0D1F').replace('\u0D28\u0D4D\u0D31', '\u0D28\u0D4D\u0D1F')
    out, pending = [], False  # pending = a consonant is waiting for its inherent 'a'
    for ch in text:
        off = _indic_offset(ch)
        if off is None:
            if pending:
                out.append('a'); pending = False
            out.append(ch)
            continue
        if off in _CONS:
            if pending:
                out.append('a')
            out.append(_CONS[off]); pending = True
        elif off in _MATRA:
            out.append(_MATRA[off]); pending = False
        elif off == _VIRAMA:
            pending = False
        elif off == _NUKTA:
            continue
        elif off in _VOWEL:
            if pending:
                out.append('a'); pending = False
            out.append(_VOWEL[off])
        elif off in _SIGN:
            if pending:
                out.append('a'); pending = False
            out.append(_SIGN[off])
        elif off in _SKIP:
            continue
        elif ch >= '\u0D00' and ch < '\u0D80' and off in _CHILLU:
            if pending:
                out.append('a')
            out.append(_CHILLU[off]); pending = False
        elif 0x66 <= off <= 0x6F:  # native digits
            if pending:
                out.append('a'); pending = False
            out.append(str(off - 0x66))
        else:
            if pending:
                out.append('a'); pending = False
    if pending:
        out.append('a')
    s = ''.join(out)
    # schwa deletion at word end ("kanstrakshana" -> "kanstrakshan")
    return re.sub(r'(?<=[bcdfghjklmnpqrstvwxyz])a\b', '', s)


def has_indic(text: str) -> bool:
    return any(_indic_offset(c) is not None for c in text or '')


# --------------------------------------------------------------------------------------
# 2. Generic cleaning
# --------------------------------------------------------------------------------------
def strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))


def base_clean(s: str) -> str:
    s = transliterate(s or '')
    s = strip_accents(s).lower()
    s = s.replace('&', ' and ').replace('+', ' and ')
    s = re.sub(r"[’'`]", '', s)          # o'brien -> obrien
    s = re.sub(r'[^a-z0-9/\-\s.,#]', ' ', s)
    return s


# --------------------------------------------------------------------------------------
# 3. Business names
# --------------------------------------------------------------------------------------
NAME_ABBR = {
    'corp': 'corporation', 'co': 'company', 'inc': 'incorporated', 'ltd': 'limited',
    'pvt': 'private', 'pvtltd': 'private limited', 'intl': 'international', 'mfg': 'manufacturing',
    'svcs': 'services', 'svc': 'service', 'assn': 'association', 'assoc': 'associates',
    'bros': 'brothers', 'dept': 'department', 'univ': 'university', 'natl': 'national',
    'mgmt': 'management', 'tech': 'technologies', 'grp': 'group', 'ent': 'enterprises',
    'cie': 'compagnie', 'ste': 'societe', 'sté': 'societe', 'fr': 'freres',
}
# legal forms / honorifics / generic fillers -> removed from the "core" name
LEGAL = {
    'incorporated', 'corporation', 'company', 'limited', 'private', 'llc', 'llp', 'pllc', 'plc',
    'lp', 'ltd', 'inc', 'pvt', 'corp', 'co', 'the', 'and', 'of', 'dba', 'mr', 'mrs', 'ms', 'dr',
    'sri', 'shri', 'shree', 'm/s', 'ms.', 'sarl', 'sas', 'sasu', 'eurl', 'sa', 'sci', 'snc',
    'scop', 'selarl', 'gmbh', 'ei', 'cie', 'compagnie', 'de', 'du', 'des', 'la', 'le', 'les', 'et', 'l', 'd', 'pa', 'pc',
}
_NAME_NOISE = [
    (re.compile(r'\((?:id|ref|no)[:\s#]*[\w-]+\)', re.I), ' '),  # "(ID: 52838)"
    (re.compile(r'#\s*\d+'), ' '),                               # "#30304"
    (re.compile(r'\|.*$'), ' '),                                 # "| www.site.com"
    (re.compile(r'\b(?:www\.)?[a-z0-9-]+\.(?:com|in|net|org|co|fr|io|biz)\b', re.I), ' '),
    (re.compile(r'\b(l)\.(l)\.(c)\.?', re.I), 'llc'),
    (re.compile(r'\b(l)\.(l)\.(p)\.?', re.I), 'llp'),
    (re.compile(r'\b(s)\.(a)\.(s)\.?', re.I), 'sas'),
    (re.compile(r'\b(e)\.(u)\.(r)\.(l)\.?', re.I), 'eurl'),
    (re.compile(r'\b(s)\.(a)\.(r)\.(l)\.?', re.I), 'sarl'),
]
_DOMAIN = re.compile(r'(?:www\.)?([a-z0-9-]+)\.(?:com|in|net|org|co|fr|io|biz)\b', re.I)
_HANDLE = re.compile(r'^[#@]([a-z0-9_]+)$', re.I)


# canonical legal-form classes (sibling distractors often differ ONLY here: "Pvt Ltd" vs "LLP")
LEGAL_CLASS = {
    'llc': 'llc', 'incorporated': 'inc', 'inc': 'inc', 'corporation': 'corp', 'corp': 'corp',
    'limited': 'ltd', 'ltd': 'ltd', 'private': 'pvt', 'pvt': 'pvt', 'llp': 'llp', 'pllc': 'pllc',
    'plc': 'plc', 'lp': 'lp', 'pc': 'pc', 'pa': 'pa', 'company': 'co', 'co': 'co', 'public': 'public',
    'sarl': 'sarl', 'sas': 'sas', 'sasu': 'sasu', 'eurl': 'eurl', 'sa': 'sa', 'sci': 'sci', 'snc': 'snc',
    'gmbh': 'gmbh', 'ei': 'ei', 'prvt': 'pvt', 'lmtd': 'ltd', 'krprshn': 'corp', 'krprsn': 'corp', 'kmpn': 'co',
}


def legal_forms(name_full: str) -> str:
    out = set()
    for t in name_full.split():
        c = LEGAL_CLASS.get(t) or LEGAL_CLASS.get(phonetic(t))
        if c:
            out.add(c)
    return ' '.join(sorted(out))


LEGAL_PH = {'prvt', 'lmtd', 'lmt', 'nkrprtd', 'krprshn', 'krprsn', 'kmpn', 'lmtdd'}


def normalize_name(raw: str):
    """Return dict of name views."""
    raw = raw or ''
    low = strip_accents(transliterate(raw)).lower().strip()
    dom = _DOMAIN.search(low)
    hnd = _HANDLE.match(low)
    domain = (dom.group(1) if dom else (hnd.group(1) if hnd else '')).replace('-', '')
    s = low
    for pat, rep in _NAME_NOISE:
        s = pat.sub(rep, s)
    s = base_clean(s)
    s = re.sub(r'[.,/\-#]', ' ', s)
    raw_toks = s.split()
    # glue runs of single letters: "l l c" -> "llc", "s a" -> "sa", "m p" -> "mp"
    glued, buf = [], ''
    for t in raw_toks:
        if len(t) == 1 and t.isalpha():
            buf += t
        else:
            if buf: glued.append(buf); buf = ''
            glued.append(t)
    if buf: glued.append(buf)
    toks = [NAME_ABBR.get(t, t) for t in glued]
    toks = ' '.join(toks).split()
    full = ' '.join(toks)
    core_toks = [t for t in toks if t not in LEGAL and not t.isdigit() and phonetic(t) not in LEGAL_PH]
    # de-duplicate consecutive repeats ("vidyalaya vidyalaya")
    core_toks = [t for i, t in enumerate(core_toks) if i == 0 or t != core_toks[i - 1]]
    core = ' '.join(core_toks)
    return {
        'name_full': full,
        'name_core': core,
        'name_sorted': ' '.join(sorted(core_toks)),     # order-invariant view
        'name_phon': ' '.join(phonetic(t) for t in core_toks),
        'name_concat': ''.join(core_toks),              # for domain-style names
        'domain': domain,
        'is_domain': int(bool(domain) and len(core_toks) == 0),
        'name_native': int(has_indic(raw)),
        'legal': legal_forms(full),
    }


# --------------------------------------------------------------------------------------
# 4. Phonetic skeleton (script-independent) -- makes "Construction" == "कंस्ट्रक्शन"
# --------------------------------------------------------------------------------------
_PH_RULES = [
    (r'tion', 'shn'), (r'sion', 'shn'), (r'ph', 'f'), (r'ck', 'k'), (r'q', 'k'), (r'x', 'ks'),
    (r'c(?=[eiy])', 's'), (r'c', 'k'), (r'w', 'v'), (r'z', 'j'),
    (r'(?<=[kgcjtdpb])h', ''), (r'sh', 's'), (r'ee', 'i'), (r'oo', 'u'),
]
_PH_RULES = [(re.compile(a), b) for a, b in _PH_RULES]


def phonetic(tok: str) -> str:
    t = tok
    for pat, rep in _PH_RULES:
        t = pat.sub(rep, t)
    head, tail = t[:1], re.sub(r'[aeiouy]', '', t[1:])
    t = (head if head not in 'aeiouy' else '') + tail
    return re.sub(r'(.)\1+', r'\1', t) or tok[:1]


# --------------------------------------------------------------------------------------
# 5. Addresses
# --------------------------------------------------------------------------------------
ADDR_ABBR = {
    'rd': 'road', 'st': 'street', 'str': 'street', 'ave': 'avenue', 'av': 'avenue',
    'blvd': 'boulevard', 'bd': 'boulevard', 'dr': 'drive', 'ln': 'lane', 'ct': 'court',
    'cir': 'circle', 'trl': 'trail', 'hwy': 'highway', 'pkwy': 'parkway', 'pl': 'place',
    'sq': 'square', 'ter': 'terrace', 'cres': 'crescent', 'apt': 'unit', 'ste': 'unit',
    'suite': 'unit', 'apartment': 'unit', 'fl': 'floor', 'flr': 'floor', 'bldg': 'building',
    'n': 'north', 's': 'south', 'e': 'east', 'w': 'west', 'ne': 'northeast', 'nw': 'northwest',
    'se': 'southeast', 'sw': 'southwest', 'mt': 'mount', 'ft': 'fort',
    'r': 'rue', 'ch': 'chemin', 'imp': 'impasse', 'all': 'allee', 'pte': 'porte',
    'opp': 'opposite', 'nr': 'near', 'nagr': 'nagar', 'mg': 'mahatma gandhi',
}
ADDR_STOP = {'no', 'door', 'h', 'hno', 'house', 'plot', 'unit', 'po', 'box', 'null', 'na',
             'c/o', 'co', 'near', 'opposite', 'and', 'the', 'of', 'de', 'du', 'des', 'la',
             'le', 'kh', 'sno', 'gala', 'shop', 'flat', 'floor', 'building', 'region'}
_NUM = re.compile(r'\d+[a-z]?(?:/\d+[a-z]?)*')


def normalize_address(raw: str):
    raw = raw or ''
    s = base_clean(raw)
    parts = [p.strip() for p in s.split(',') if p.strip()]
    s2 = re.sub(r'[.#/\-]', ' ', s.replace(',', ' , '))
    toks = [ADDR_ABBR.get(t, t) for t in s2.split() if t != ',']
    toks = ' '.join(toks).split()
    content = [t for t in toks if t not in ADDR_STOP]
    nums = _NUM.findall(s)
    nums = [n for n in nums if len(n) <= 8]
    words = [t for t in content if not t[0].isdigit()]
    return {
        'addr_full': ' '.join(toks),
        'addr_words': ' '.join(words),
        'addr_nums': ' '.join(nums),
        'house_no': nums[0] if nums else '',
        'postcode': next((n for n in nums if len(n) in (5, 6) and n.isdigit()), ''),
        'addr_parts': parts,
        'addr_empty': int(len(raw.strip()) == 0),
        'addr_native': int(has_indic(raw)),
    }


if __name__ == '__main__':
    for n in ['ईस्टर्न कंस्ट्रक्शन प्राइवेट लिमिटेड', 'Eastern Construction Private Limited',
              'গ্রেট অ্যাগ্রো লিমিটেড', 'Great Agro Limited', 'Clean  Highland Forefront LLC (ID: 52838)',
              'ablecarleykeo.com', '#shieldscordova', 'WHITTLE, OLIVAS and MITCHELL L.L.C.',
              'Europ & Frères Distribution S.A.', 'SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPORATION | www.shivshakti.com']:
        print(n, '->', normalize_name(n))
    for a in ['NO ##701 RATNESHWAR BLDG268 BHAGWANDAS INDRAJIT RD BANGANGA, MUMBAI, महाराष्ट्र',
              '309-313 RATTVIK CIR, BUFFALO, MN', '63 R. DE DIEPPE, LILLE, Hauts-de-France']:
        print(a, '->', normalize_address(a))


def street_and_locality(addr_parts):
    """Split a normalised address into the street part (the comma-part holding the first
    number, else the first part) and the locality words (all other parts)."""
    if not addr_parts:
        return '', ''
    si = next((i for i, p in enumerate(addr_parts) if any(ch.isdigit() for ch in p)), 0)
    def words(p):
        ts = re.sub(r'[.#/\-]', ' ', p).split()
        ts = [ADDR_ABBR.get(t, t) for t in ts]
        return [t for t in ' '.join(ts).split() if t.isalpha() and len(t) > 1 and t not in ADDR_STOP]
    street = words(addr_parts[si])
    loc = [w for i, p in enumerate(addr_parts) if i != si for w in words(p)]
    return ' '.join(street), ' '.join(loc)


In [ ]:
%%writefile {SRC}/blocking.py
"""
Candidate generation (blocking).

Several cheap, complementary "views" each propose top-K Source-2/3 neighbours for every
Source-1 record, *within the same country string* (open set: whatever labels appear).
The union of passes is the candidate set.

Each view = char n-gram TF-IDF  ->  TruncatedSVD (dense, d dims)  ->  L2-normalised
so that the inner product approximates TF-IDF cosine similarity. At full scale the dense
vectors go into FAISS (MIT licence); here a chunked numpy matmul does exact top-K.
"""
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize as l2norm

VIEWS = {
    # name only: core + phonetic skeleton + domain/concat (catches native script & URLs)
    'name': lambda d: (d.name_core + ' | ' + d.name_phon + ' | ' + d.domain),
    # address only: catches "different trade name, same premises"
    'addr': lambda d: d.addr_full,
    # joint view
    'both': lambda d: (d.name_core + ' | ' + d.name_phon + ' | ' + d.addr_words + ' ' + d.addr_nums),
}


def _embed(texts_fit, texts_a, texts_b, dim=192, seed=0, fit_n=200_000):
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=2, sublinear_tf=True,
                          dtype=np.float32, max_features=300_000)
    rng = np.random.default_rng(seed)
    fit = texts_fit if len(texts_fit) <= fit_n else texts_fit.iloc[rng.choice(len(texts_fit), fit_n, replace=False)]
    vec.fit(fit)
    A, B = vec.transform(texts_a), vec.transform(texts_b)
    svd = TruncatedSVD(n_components=min(dim, A.shape[1] - 1), random_state=seed, algorithm='randomized', n_iter=4)
    svd.fit(vec.transform(fit))
    ea = l2norm(svd.transform(A)).astype(np.float32)
    eb = l2norm(svd.transform(B)).astype(np.float32)
    return ea, eb


def topk_dense(ea, eb, k, chunk=512):
    """Exact inner-product top-k of each row of ea against eb. Returns (idx, sim)."""
    k = min(k, eb.shape[0])
    I = np.empty((ea.shape[0], k), dtype=np.int64)
    S = np.empty((ea.shape[0], k), dtype=np.float32)
    for i in range(0, ea.shape[0], chunk):
        sims = ea[i:i + chunk] @ eb.T
        part = np.argpartition(-sims, k - 1, axis=1)[:, :k]
        ps = np.take_along_axis(sims, part, axis=1)
        o = np.argsort(-ps, axis=1)
        I[i:i + chunk] = np.take_along_axis(part, o, axis=1)
        S[i:i + chunk] = np.take_along_axis(ps, o, axis=1)
    return I, S


def generate_candidates(E1: pd.DataFrame, E2: pd.DataFrame, ks=None, dim=192, backend='numpy'):
    """Return DataFrame [i1, i2, pass, rank, sim] (row positions; one row per pass hit)."""
    ks = ks or {'name': 40, 'addr': 25, 'both': 40}
    out = []
    for country in sorted(set(E1.country)):
        a = E1[E1.country == country]
        b = E2[E2.country == country]
        if len(a) == 0 or len(b) == 0:
            continue
        for view, k in ks.items():
            ta, tb = VIEWS[view](a), VIEWS[view](b)
            ea, eb = _embed(pd.concat([ta, tb]), ta, tb, dim=dim)
            I, S = topk_backend(ea, eb, min(k, len(b)), backend)
            ia = np.flatnonzero((E1.country == country).values)
            ib = np.flatnonzero((E2.country == country).values)
            out.append(pd.DataFrame({
                'i1': np.repeat(ia, I.shape[1]).astype(np.int32),   # row position in E1
                'i2': ib[I.ravel()].astype(np.int32),                # row position in E2
                'pass': np.int8(list(ks).index(view)),
                'rank': np.tile(np.arange(I.shape[1], dtype=np.int16), len(a)),
                'sim': S.ravel(),
            }))
    return pd.concat(out, ignore_index=True)


# ---------------------------------------------------------------------------------------
# Pass 4: exact-key inverted index (rare tokens).  Complements the dense passes, which
# blur rare identifiers such as "14/2995", "xii/228-d" or a 3-letter brand like "pfx".
# ---------------------------------------------------------------------------------------
import re
_TOK = re.compile(r'[a-z0-9/]+')


_DIG = re.compile(r'\d+')


def record_keys(d: pd.DataFrame):
    """List of blocking keys per record (country-prefixed)."""
    keys = []
    locs = d.locality.values if 'locality' in d.columns else None
    streets = d.street.values if ('street' in d.columns and locs is not None) else None
    for ri, (c, full, core, phon, dom, nums) in enumerate(zip(d.country.values, d.addr_full.values, d.name_core.values,
                                              d.name_phon.values, d.domain.values, d.addr_nums.values)):
        ks = set()
        at = full.split()
        for i, t in enumerate(at):
            if any(ch.isdigit() for ch in t):
                if len(t) >= 4 or '/' in t:
                    ks.add('A:' + t)
                nxt = next((w for w in at[i + 1:i + 3] if w.isalpha() and len(w) > 2), None)
                if nxt:
                    ks.add('H:' + t + '_' + nxt[:5])          # house no + street word
        for n in nums.split():
            if '/' in n or len(n) >= 4:
                ks.add('A:' + n.replace('/', ''))
        ct = core.split()
        for t in ct:
            if len(t) >= 3:
                ks.add('N:' + t)
        for a, b in zip(ct, ct[1:]):
            ks.add('B:' + a + '_' + b)
        for t in phon.split():
            if len(t) >= 3:
                ks.add('P:' + t)
        if dom:
            ks.add('D:' + dom)
        if locs is not None:            # name token scoped to the city: rare locally even if common nationally
            lw = locs[ri].split()[:2]
            for l in lw:
                for t in ct:
                    if len(t) >= 3:
                        ks.add('L:' + t + '@' + l)
                for t in phon.split():
                    if len(t) >= 3:
                        ks.add('Q:' + t + '@' + l)
        if streets is not None:         # v6: digit groups (zero-stripped), numbers / street words scoped to a city word
            g = [x.lstrip('0') or '0' for x in _DIG.findall(full)]
            for a, b in zip(g, g[1:]):
                ks.add('G:' + a + '_' + b)
            lw = list(dict.fromkeys(locs[ri].split()))[-6:]
            sw = [w for w in streets[ri].split() if len(w) >= 4]
            for l in lw:
                for x in g:
                    if len(x) >= 2:
                        ks.add('g:' + x + '@' + l)
                for w in sw:
                    if w != l:
                        ks.add('W:' + w + '@' + l)
        keys.append([c + '|' + k for k in ks])
    return keys


def key_candidates(E1, E2, k=30, max_df=40):
    k1 = pd.DataFrame({'i1': np.arange(len(E1), dtype=np.int32), 'key': record_keys(E1)}).explode('key').dropna()
    k2 = pd.DataFrame({'i2': np.arange(len(E2), dtype=np.int32), 'key': record_keys(E2)}).explode('key').dropna()
    df = k2.key.value_counts()
    ok = df[df <= max_df]
    k2 = k2[k2.key.isin(ok.index)]
    k2 = k2.assign(w=np.log1p(len(E2) / k2.key.map(ok).values)).astype({'w': np.float32})
    m = k1.merge(k2, on='key')
    g = m.groupby(['i1', 'i2'], sort=False).w.sum().reset_index()
    g = g.sort_values(['i1', 'w'], ascending=[True, False])
    g['rank'] = g.groupby('i1').cumcount().astype(np.int16)
    g = g[g['rank'] < k]
    return pd.DataFrame({'i1': g.i1.values, 'i2': g.i2.values, 'pass': np.int8(3),
                         'rank': g['rank'].values, 'sim': g.w.values})


PASS_NAMES = ['name', 'addr', 'both', 'key']


def build_candidates(E1, E2, ks=None, key_k=20, key_max_df=40, dim=192, backend='numpy'):
    """Run all passes, union them, and pivot per-pass (sim, rank) into columns."""
    ks = ks or {'name': 20, 'addr': 10, 'both': 20}
    D = generate_candidates(E1, E2, ks=ks, dim=dim, backend=backend)
    K = key_candidates_np(E1, E2, k=key_k, max_df=key_max_df)
    R = pd.concat([D, K], ignore_index=True)
    R['pname'] = np.array(PASS_NAMES)[R['pass'].values]
    sim = R.pivot_table(index=['i1', 'i2'], columns='pname', values='sim', aggfunc='max')
    rank = R.pivot_table(index=['i1', 'i2'], columns='pname', values='rank', aggfunc='min')
    sim.columns = ['sim_' + c for c in sim.columns]
    rank.columns = ['rank_' + c for c in rank.columns]
    C = pd.concat([sim, rank], axis=1)
    C['n_passes'] = sim.notna().sum(axis=1)
    C = C.fillna(-1).reset_index()
    return C


# =======================================================================================
# Full-scale variants (10M+ records): hashed integer keys + numpy sort-merge join, and a
# pluggable top-K backend (numpy on CPU, torch on GPU, or FAISS IVF).
# =======================================================================================
def _hash_keys(keys_per_record):
    """list[list[str]] -> (row_idx int32[], key int64[])"""
    rows, ks = [], []
    for r, kl in enumerate(keys_per_record):
        for k in kl:
            rows.append(r); ks.append(hash(k))
    return np.asarray(rows, dtype=np.int32), np.asarray(ks, dtype=np.int64)


def _hash_keys_chunked(d, chunk=300_000):
    """(row, key-hash) arrays built 300k records at a time (a full key list of 6M records
    would be ~15 GB of Python strings)."""
    R, H = [], []
    for s in range(0, len(d), chunk):
        r, h = _hash_keys(record_keys(d.iloc[s:s + chunk])); R.append(r + s); H.append(h)
    return (np.concatenate(R), np.concatenate(H)) if R else (np.zeros(0, np.int32), np.zeros(0, np.int64))


def key_candidates_np(E1, E2, k=20, max_df=40, chunk=200_000):
    """Same scoring as key_candidates() but memory-lean: no string joins."""
    r2, h2 = _hash_keys_chunked(E2)
    o = np.argsort(h2, kind='stable'); r2, h2 = r2[o], h2[o]
    uk, start, cnt = np.unique(h2, return_index=True, return_counts=True)
    w_key = np.log1p(len(E2) / cnt).astype(np.float32)
    keep = cnt <= max_df
    uk, start, cnt, w_key = uk[keep], start[keep], cnt[keep], w_key[keep]
    out = []
    for s in range(0, len(E1), chunk):
        r1, h1 = _hash_keys(record_keys(E1.iloc[s:s + chunk])); r1 += s
        pos = np.searchsorted(uk, h1)
        pos[pos >= len(uk)] = 0
        hit = uk[pos] == h1
        r1, pos = r1[hit], pos[hit]
        n = cnt[pos]
        rep_r1 = np.repeat(r1, n)
        rep_w = np.repeat(w_key[pos], n)
        offs = np.repeat(start[pos] - np.cumsum(n) + n, n) + np.arange(n.sum())
        rep_r2 = r2[offs]
        code = rep_r1.astype(np.int64) << 32 | rep_r2.astype(np.int64)
        uc, inv = np.unique(code, return_inverse=True)
        score = np.bincount(inv, weights=rep_w).astype(np.float32)
        i1 = (uc >> 32).astype(np.int32); i2 = (uc & 0xFFFFFFFF).astype(np.int32)
        o = np.lexsort((-score, i1))
        i1, i2, score = i1[o], i2[o], score[o]
        first = np.r_[0, np.flatnonzero(np.diff(i1)) + 1]
        rank = np.arange(len(i1)) - np.repeat(first, np.diff(np.r_[first, len(i1)]))
        m = rank < k
        out.append(pd.DataFrame({'i1': i1[m], 'i2': i2[m], 'pass': np.int8(3),
                                 'rank': rank[m].astype(np.int16), 'sim': score[m]}))
    return pd.concat(out, ignore_index=True)


def topk_backend(ea, eb, k, backend='numpy'):
    """Top-K inner product search. 'torch' uses a GPU if present (fp16 matmul);
    'faiss' uses an IVF-Flat index (approximate, CPU or GPU)."""
    if backend == 'torch':
        import torch
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        dt = torch.float16 if dev == 'cuda' else torch.float32
        B = torch.from_numpy(eb).to(dev, dt)
        I = np.empty((len(ea), k), np.int64); S = np.empty((len(ea), k), np.float32)
        for s in range(0, len(ea), 4096):
            A = torch.from_numpy(ea[s:s + 4096]).to(dev, dt)
            v, i = torch.topk(A @ B.T, k, dim=1)
            I[s:s + 4096] = i.cpu().numpy(); S[s:s + 4096] = v.float().cpu().numpy()
        return I, S
    if backend == 'faiss':
        import faiss
        d = eb.shape[1]
        nlist = int(max(64, min(65536, 4 * np.sqrt(len(eb)))))
        q = faiss.IndexFlatIP(d)
        index = faiss.IndexIVFFlat(q, d, nlist, faiss.METRIC_INNER_PRODUCT)
        index.train(eb[np.random.default_rng(0).choice(len(eb), min(len(eb), 50 * nlist), replace=False)])
        index.add(eb)
        index.nprobe = 32
        S, I = index.search(ea, k)
        return I, S
    return topk_dense(ea, eb, k)


In [ ]:
%%writefile {SRC}/decide.py
"""
Turn pair probabilities into the final match lists.

1. One-owner constraint: every S2/S3 record keeps only its best-scoring S1.
2. Per-S1 subset choice, either
   a) global threshold t (tuned for macro F0.5 on validation), or
   b) expected-F0.5 maximisation (plug-in approximation, see docs).
"""
import numpy as np
import pandas as pd


def exclusive(d: pd.DataFrame) -> pd.DataFrame:
    """Keep, for each candidate i2, only the row with max p (ties -> first)."""
    idx = d.groupby('i2').p.idxmax()
    return d.loc[idx]


def by_threshold(d: pd.DataFrame, t: float):
    return d[d.p >= t]


def by_expected_f(d: pd.DataFrame, beta=0.5, floor=0.05):
    """For each S1 choose k maximising E[F_beta] ~ (1+b2) sum_{j<=k} p_j / (b2*sum_all p + k),
    compared with the empty set whose expected score is P(no true match) = prod(1-p_j)."""
    b2 = beta * beta
    d = d[d.p >= floor].sort_values(['i1', 'p'], ascending=[True, False])
    keep = []
    for i1, g in d.groupby('i1', sort=False):
        p = g.p.values
        csum = np.cumsum(p)
        k = np.arange(1, len(p) + 1)
        ef = (1 + b2) * csum / (b2 * p.sum() + k)
        best = ef.argmax()
        if ef[best] > np.prod(1 - p):
            keep.append(g.index.values[:best + 1])
    return d.loc[np.concatenate(keep)] if keep else d.iloc[:0]


def to_sets(d: pd.DataFrame, ids1, ids2):
    out = {}
    for i1, g in d.groupby('i1'):
        out[ids1[i1]] = set(ids2[g.i2.values])
    return out


In [ ]:
%%writefile {SRC}/metric.py
"""Official metric: macro-averaged F0.5 over Source-1 entities (singletons included)."""
import numpy as np


def f_beta_entity(pred: set, true: set, beta=0.5):
    if not true and not pred:
        return 1.0
    if not true or not pred:
        return 0.0
    tp = len(pred & true)
    if tp == 0:
        return 0.0
    p, r = tp / len(pred), tp / len(true)
    b2 = beta * beta
    return (1 + b2) * p * r / (b2 * p + r)


def macro_f05(pred: dict, truth: dict):
    """pred/truth: {s1_id: set(ids)}; averaged over the keys of `truth`."""
    return float(np.mean([f_beta_entity(pred.get(k, set()), v) for k, v in truth.items()]))


if __name__ == '__main__':
    # example from the problem statement -> 0.714
    print(round(f_beta_entity({'a', 'b', 'c'}, {'a', 'c'}), 3))


In [ ]:
%%writefile {SRC}/fasttext_feats.py
"""Vectorised (numpy) hashed char n-gram TF-IDF: replaces sklearn's pure-Python analyzers."""
import numpy as np, scipy.sparse as sp

_P = np.uint32(16777619)            # FNV-1a 32-bit prime
_SALT = {1: 0x9E3779B9, 2: 0xC2B2AE3D, 3: 0x165667B1, 4: 0x27D4EB2F, 5: 0x94D049BB, 6: 0xBF58476D}


def _pad(texts, wb):
    if wb:  # char_wb semantics: every word padded by one space, n-grams never span words
        return [' ' + '  '.join(t.split()) + ' ' if t else '' for t in texts]
    return [' ' + t + ' ' if t else '' for t in texts]


def char_counts(texts, ns=(2, 3, 4), n_features=1 << 20, wb=True, chunk=100_000, maxlen=256):
    """CSR (float32 counts) of hashed char n-grams. Rows are processed in length-sorted
    chunks so the padded byte matrix stays tight."""
    pt = _pad(list(texts), wb)
    N = len(pt)
    lens = np.fromiter((len(t) for t in pt), np.int32, N)
    order = np.argsort(lens, kind='stable')
    mask = np.uint32(n_features - 1)
    R, Cc = [], []
    for s in range(0, N, chunk):
        idx = order[s:s + chunk]
        L = int(min(maxlen, max(1, lens[idx].max())))
        arr = np.array([pt[i][:L].encode('ascii', 'ignore') for i in idx], dtype=f'S{L}')
        B = np.frombuffer(arr.tobytes(), np.uint8).reshape(len(idx), L)
        for n in ns:
            if L < n:
                continue
            W = L - n + 1
            h = np.full((len(idx), W), _SALT[n], np.uint32)
            valid = B[:, 0:W] != 0
            for j in range(n):
                cj = B[:, j:j + W]
                h ^= cj
                h *= _P
                if j:
                    valid &= cj != 0
                if wb and 0 < j < n - 1:
                    valid &= cj != 32
            if wb and n == 2:
                valid &= ~((B[:, 0:W] == 32) & (B[:, 1:1 + W] == 32))
            r, c = np.nonzero(valid)
            R.append(idx[r]); Cc.append(h[r, c] & mask)
    r = np.concatenate(R) if R else np.zeros(0, np.int64)
    c = np.concatenate(Cc).astype(np.int32) if Cc else np.zeros(0, np.int32)
    m = sp.csr_matrix((np.ones(len(r), np.float32), (r, c)), shape=(N, n_features))
    m.sum_duplicates()
    return m


class HashTfidf:
    """TF-IDF over hashed char n-grams; idf fitted on a sample. transform() is vectorised."""
    def __init__(self, ns=(2, 3, 4), n_features=1 << 20, wb=True, sublinear=True):
        self.ns, self.nf, self.wb, self.sub = ns, n_features, wb, sublinear

    def fit(self, texts):
        X = char_counts(texts, self.ns, self.nf, self.wb)
        df = np.bincount(X.indices, minlength=self.nf)
        self.idf = (np.log((1 + X.shape[0]) / (1 + df)) + 1).astype(np.float32)
        return self

    def transform(self, texts, norm=True):
        X = char_counts(texts, self.ns, self.nf, self.wb)
        if self.sub:
            X.data = 1 + np.log(X.data)
        X.data *= self.idf[X.indices]
        if norm:
            n = np.sqrt(np.asarray(X.multiply(X).sum(1)).ravel()); n[n == 0] = 1
            X = sp.diags(1 / n).dot(X).tocsr()
        return X.astype(np.float32)


In [ ]:
%%writefile {SRC}/er_fast.py
"""
er_fast — Kaggle-optimised entity-resolution pipeline (4 vCPU, ~30 GB RAM, 2 x T4).

What changed vs. the first Kaggle version (and why)
---------------------------------------------------
* Fine-grained checkpoints: stage 1 per 400k-row chunk, stage 2 per view, stage 3 per
  feature column, stage 4 per model, stage 5 per country. A restart loses minutes, not hours.
* Column store: every normalised frame is saved one column per file, so each step loads only
  the columns it needs (blocking needs 8 of 25 -> ~1/3 of the RAM).
* Vectorised text features: char n-grams are hashed with numpy (no per-string Python), word
  tokens with pandas' C hashing; ~3-10x faster than sklearn analyzers, no vocabulary to ship.
* numba kernels (4 cores, parallel) for sparse row-wise dot products of 100M+ pairs.
* GPU everywhere it pays: SVD projection with torch on GPU, FAISS exact search sharded over
  both T4s running in a background thread while the CPU embeds the next view, XGBoost with
  the two stage-A models trained concurrently on GPU 0 and GPU 1.
* Worker processes start clean (forkserver) and a dead/stuck worker times out instead of
  hanging the kernel.
All accelerations fall back to plain CPU code if a library is missing.
"""
import os, sys, gc, time, json, pickle, zlib, glob, threading
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from concurrent.futures.process import BrokenProcessPool
from concurrent.futures import TimeoutError as FuturesTimeout
import numpy as np
import pandas as pd
import scipy.sparse as sp

from normalize import normalize_name, normalize_address, street_and_locality
from blocking import VIEWS, record_keys, key_candidates_np, topk_dense
from decide import exclusive, by_threshold, by_expected_f, to_sets
from metric import macro_f05
from fasttext_feats import char_counts

T0 = time.time()
N_JOBS = max(1, min(32, os.cpu_count() or 1))   # >32 workers only adds RAM/IPC overhead


def _mem():
    try:
        rss = int([l for l in open('/proc/self/status') if l.startswith('VmRSS')][0].split()[1]) / 1e6
        av = int([l for l in open('/proc/meminfo') if l.startswith('MemAvailable')][0].split()[1]) / 1e6
        return f'[rss {rss:.1f}G free {av:.0f}G]'
    except Exception:
        return ''


def log(*a):
    m, s = divmod(int(time.time() - T0), 60); h, m = divmod(m, 60)
    print(f'[{h:d}:{m:02d}:{s:02d}]{_mem()}', *a, flush=True)


def n_gpus():
    try:
        import torch
        return torch.cuda.device_count()
    except Exception:
        return 0


# ============================================================================ utilities
def _pmap(fn, items, timeout=None):
    """Parallel map in clean forkserver workers; dead or stuck workers -> single-process redo."""
    if N_JOBS == 1 or len(items) <= 1:
        return [fn(x) for x in items]
    ex = ProcessPoolExecutor(max_workers=N_JOBS, mp_context=mp.get_context('forkserver'))
    try:
        return list(ex.map(fn, items, timeout=timeout or 900 + 180 * len(items)))
    except (BrokenProcessPool, FuturesTimeout, TimeoutError) as e:
        log(f'WARNING: worker pool failed ({type(e).__name__}) -> redoing in one process')
        for p in list((getattr(ex, '_processes', None) or {}).values()):
            try:
                p.kill()
            except Exception:
                pass
        gc.collect()
        return [fn(x) for x in items]
    finally:
        ex.shutdown(wait=False, cancel_futures=True)


def read_tsv(path, **kw):
    # encoding_errors='replace': one corrupt byte (e.g. a truncated copy) must not kill a 5M-row read
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False, quoting=3,
                       encoding='utf-8', encoding_errors='replace', **kw)


def find_data_dir(root='/kaggle/input'):
    for dp, _, fs in os.walk(root):
        if 'train_source1.tsv' in fs:
            return os.path.dirname(dp)
    raise FileNotFoundError('train_source1.tsv not found under ' + root)


class ColStore:
    """One pickle per column -> load only what a step needs."""
    @staticmethod
    def save(d, df):
        os.makedirs(d, exist_ok=True)
        for c in df.columns:
            df[c].reset_index(drop=True).to_pickle(f'{d}/{c}.pkl')
        json.dump({'n': len(df), 'cols': list(df.columns)}, open(f'{d}/_meta.json', 'w'))

    @staticmethod
    def load(d, cols=None):
        meta = json.load(open(f'{d}/_meta.json'))
        cols = cols or meta['cols']
        return pd.DataFrame({c: pd.read_pickle(f'{d}/{c}.pkl') for c in cols})

    @staticmethod
    def n(d):
        return json.load(open(f'{d}/_meta.json'))['n']

    @staticmethod
    def exists(d):
        return os.path.exists(f'{d}/_meta.json')


def O(s):
    return s.to_numpy(dtype=object)


# ============================================================================ stage 1: normalise
KEEP = ['entity_id', 'country', 'src', 'name_full', 'name_core', 'name_sorted', 'name_phon',
        'name_concat', 'domain', 'is_domain', 'name_native', 'legal', 'addr_full', 'addr_words',
        'addr_nums', 'house_no', 'addr_empty', 'addr_native', 'street', 'locality',
        'name_distinct', 'house_dig', 'num_tok', 'first_tok', 'n_len']
INT_COLS = ['is_domain', 'name_native', 'addr_empty', 'addr_native', 'n_len']


def _digits(s):
    return ''.join(ch for ch in s if ch.isdigit())


def enrich(df):
    rows = []
    for eid, nm, ad, c in zip(df.entity_id.values, df.business_name.values,
                              df.business_address.values, df.country.values):
        n = normalize_name(nm); a = normalize_address(ad)
        st, lo = street_and_locality(a.pop('addr_parts'))
        bad = set(st.split()) | set(lo.split())
        core = n['name_core'].split()
        rows.append({'entity_id': eid, 'country': c, 'src': eid[:2], **n, **a, 'street': st, 'locality': lo,
                     'name_distinct': ' '.join(t for t in core if t not in bad),
                     'house_dig': _digits(a['house_no']),
                     'num_tok': a['addr_nums'].split()[0] if a['addr_nums'] else '',
                     'first_tok': core[0] if core else '', 'n_len': len(core)})
    if not rows:
        return pd.DataFrame({c: pd.Series([], dtype=object) for c in KEEP})
    out = pd.DataFrame(rows)[KEEP]
    for c in INT_COLS:
        out[c] = out[c].astype(np.int8)
    return out


def _count_rows(path):
    n, last = 0, b'\n'
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''):
            n += b.count(b'\n'); last = b[-1:]
    return n - 1 + (last != b'\n')        # a final line without newline still counts


def _data_fp(cfg, split):
    d = f"{cfg['data']}/{split}"
    return {f: [os.path.getsize(f'{d}/{f}'), int(os.path.getmtime(f'{d}/{f}'))]
            for f in sorted(os.listdir(d)) if f.endswith('.tsv')}


def stage_normalise(cfg, split):
    """Streams each source file in 400k-row chunks; every chunk is a checkpoint.
    Guards: (1) if the data files changed since the cache was built (size/mtime), the split's
    cache AND the trained models are rebuilt; (2) rows processed must equal rows in the file."""
    w = f"{cfg['work']}/{split}"
    fp_now = _data_fp(cfg, split)
    fp_file = f'{w}/data_fp.json'
    if os.path.isdir(w) and (not os.path.exists(fp_file) or json.load(open(fp_file)) != fp_now):
        import shutil
        log(split, 'WARNING: cache was built from different/older data files -> deleting it and the models')
        shutil.rmtree(w, ignore_errors=True); shutil.rmtree(f"{cfg['work']}/models", ignore_errors=True)
    os.makedirs(f'{w}/_shards', exist_ok=True)
    json.dump(fp_now, open(fp_file, 'w'))
    if os.path.exists(f'{w}/countries.json'):
        log(split, 'normalise: cached'); return json.load(open(f'{w}/countries.json'))
    d = f"{cfg['data']}/{split}"
    for src in (1, 2, 3):
        path = f'{d}/{split}_source{src}.tsv'
        for ci, ch in enumerate(read_tsv(path, chunksize=400_000)):
            tag = f'{w}/_shards/s{src}_{ci:04d}'
            if os.path.exists(tag + '.done'):
                continue
            if split == 'train' and src == 1 and cfg['train_frac'] < 1:
                keep = np.array([zlib.crc32(x.encode()) % 1000 < cfg['train_frac'] * 1000 for x in ch.entity_id])
                ch = ch[keep]
            if src == 1:
                np.save(tag + '_ids.npy', ch.entity_id.to_numpy(dtype=object), allow_pickle=True)
            parts = [ch.iloc[i:i + 50_000] for i in range(0, len(ch), 50_000)]
            e = pd.concat(_pmap(enrich, parts), ignore_index=True) if parts else enrich(ch)
            for c, g in e.groupby('country', sort=False):
                g.reset_index(drop=True).to_pickle(f'{tag}_{c}.pkl')
            open(tag + '.done', 'w').write(str(len(ch)) if not (split == 'train' and src == 1 and cfg['train_frac'] < 1) else '-1')
            log(split, f'source{src} chunk {ci} normalised ({len(ch):,} rows)')
            del e, ch; gc.collect()
    # every row of every file must have been processed (catches truncated / half-extracted files)
    for src in (1, 2, 3):
        got = [open(f).read() for f in glob.glob(f'{w}/_shards/s{src}_*.done')]
        if any(g in ('', '-1') for g in got):
            continue
        want = _count_rows(f'{d}/{split}_source{src}.tsv')
        if sum(int(g) for g in got) != want:
            raise RuntimeError(f'{split}_source{src}: processed {sum(int(g) for g in got):,} rows but the file has '
                               f'{want:,}. Delete {w} and re-run (the data files were probably re-extracted).')
    # consolidate per country into column stores
    shard_files = sorted(glob.glob(f'{w}/_shards/s*_*_*.pkl'))
    countries = sorted({os.path.basename(f).split('_', 2)[2][:-4] for f in shard_files})
    for c in countries:
        for side, srcs in (('E1', ('s1',)), ('E2', ('s2', 's3'))):
            dd = f'{w}/{c}/{side}'
            if ColStore.exists(dd):
                continue
            fs = [f for s_ in srcs for f in sorted(glob.glob(f'{w}/_shards/{s_}_*_{c}.pkl'))]
            df = pd.concat([pd.read_pickle(f) for f in fs], ignore_index=True) if fs else enrich(pd.DataFrame(
                columns=['entity_id', 'business_name', 'business_address', 'country']))
            ColStore.save(dd, df)
            np.save(f'{w}/{c}/ids{side[1]}.npy', df.entity_id.to_numpy(dtype=object), allow_pickle=True)
            del df; gc.collect()
        log(split, c, 'column store ready:', ColStore.n(f'{w}/{c}/E1'), 'S1 /', ColStore.n(f'{w}/{c}/E2'), 'S2+S3')
    ids = np.concatenate([np.load(f, allow_pickle=True) for f in sorted(glob.glob(f'{w}/_shards/s1_*_ids.npy'))])
    np.save(f'{w}/s1_order.npy', ids, allow_pickle=True)
    json.dump(countries, open(f'{w}/countries.json', 'w'))
    log(split, 'normalise done', countries)
    return countries


# ============================================================================ text vectors
class CharTfidf:
    """Hashed char n-gram TF-IDF. prune=True keeps only buckets seen >= min_df times in the fit
    sample (compact columns, needed before SVD); prune=False keeps all 2^20 buckets."""
    def __init__(self, ns=(2, 3, 4), wb=True, nf=1 << 20, prune=False, min_df=2):
        self.ns, self.wb, self.nf, self.prune, self.min_df = ns, wb, nf, prune, min_df

    def fit(self, texts, n=400_000):
        t = list(texts)
        if len(t) > n:
            t = [t[i] for i in np.random.default_rng(0).choice(len(t), n, replace=False)]
        X = char_counts(t, self.ns, self.nf, self.wb)
        df = np.bincount(X.indices, minlength=self.nf)
        self.idf = (np.log((1 + X.shape[0]) / (1 + df)) + 1).astype(np.float32)
        if self.prune:
            keep = df >= self.min_df
            self.colmap = np.full(self.nf, -1, np.int32); self.colmap[keep] = np.arange(keep.sum())
            self.idf = self.idf[keep]; self.dim = int(keep.sum())
        return self

    def _one(self, texts):
        X = char_counts(texts, self.ns, self.nf, self.wb)
        X.data = 1 + np.log(X.data)
        if self.prune:
            c = self.colmap[X.indices]; ok = c >= 0
            rows = np.repeat(np.arange(X.shape[0]), np.diff(X.indptr))[ok]
            X = sp.csr_matrix((X.data[ok], (rows, c[ok])), shape=(X.shape[0], self.dim))
        X.data *= self.idf[X.indices]
        n = np.sqrt(np.asarray(X.multiply(X).sum(1)).ravel()); n[n == 0] = 1
        X = sp.diags((1 / n).astype(np.float32)).dot(X).tocsr()
        X.sort_indices()
        return X.astype(np.float32)

    def transform(self, texts, chunk=250_000):
        t = list(texts)
        parts = [t[i:i + chunk] for i in range(0, len(t), chunk)]
        if len(parts) <= 1:
            return self._one(t)
        return sp.vstack(_pmap(_CharJob(self), parts), format='csr')


class _CharJob:
    def __init__(self, v): self.v = v
    def __call__(self, t): return self.v._one(t)


def word_counts(texts, ngram=1, nf=1 << 22, binary=True):
    """Whitespace tokens (+ adjacent bigrams if ngram=2) hashed with pandas' C hash -> CSR."""
    s = pd.Series(list(texts), dtype=object)
    tok = s.str.split()
    ex = tok.explode()
    ex = ex[ex.notna() & (ex != '')]
    rows = ex.index.to_numpy()
    vals = ex.to_numpy(dtype=object)
    if ngram == 2 and len(vals) > 1:
        same = rows[1:] == rows[:-1]
        big = (pd.Series(vals[:-1][same]) + ' ' + pd.Series(vals[1:][same])).to_numpy(dtype=object)
        rows = np.concatenate([rows, rows[:-1][same]]); vals = np.concatenate([vals, big])
    if len(vals) == 0:
        return sp.csr_matrix((len(s), nf), dtype=np.float32)
    cols = (pd.util.hash_array(vals) & np.uint64(nf - 1)).astype(np.int32)
    X = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(len(s), nf))
    X.sum_duplicates()
    if binary:
        X.data[:] = 1
    X.sort_indices()
    return X


class WordTfidf:
    def __init__(self, ngram=1, nf=1 << 22):
        self.ngram, self.nf = ngram, nf

    def fit(self, texts, n=1_000_000):
        t = list(texts)
        if len(t) > n:
            t = [t[i] for i in np.random.default_rng(0).choice(len(t), n, replace=False)]
        X = word_counts(t, self.ngram, self.nf, binary=True)
        df = np.bincount(X.indices, minlength=self.nf)
        self.idf = (np.log((1 + X.shape[0]) / (1 + df)) + 1).astype(np.float32)
        return self

    def transform(self, texts):
        X = word_counts(texts, self.ngram, self.nf, binary=False)
        X.data = (1 + np.log(X.data)) * self.idf[X.indices]
        n = np.sqrt(np.asarray(X.multiply(X).sum(1)).ravel()); n[n == 0] = 1
        X = sp.diags((1 / n).astype(np.float32)).dot(X).tocsr()
        X.sort_indices()
        return X.astype(np.float32)


# numba row-wise sparse dot product (parallel over pairs); scipy fallback
try:
    from numba import njit, prange

    @njit(parallel=True, fastmath=True, cache=True)
    def _rowdot_nb(ap, ai, av, bp, bi, bv, ia, ib, out):
        for k in prange(ia.shape[0]):
            p = ap[ia[k]]; pe = ap[ia[k] + 1]; q = bp[ib[k]]; qe = bp[ib[k] + 1]
            s = 0.0
            while p < pe and q < qe:
                x = ai[p]; y = bi[q]
                if x == y:
                    s += av[p] * bv[q]; p += 1; q += 1
                elif x < y:
                    p += 1
                else:
                    q += 1
            out[k] = s
    HAVE_NUMBA = True
except Exception:
    HAVE_NUMBA = False


def rowdot(A, B, ia, ib, chunk=300_000):
    out = np.empty(len(ia), np.float32)
    if HAVE_NUMBA:
        try:
            _rowdot_nb(A.indptr.astype(np.int64), A.indices, A.data, B.indptr.astype(np.int64), B.indices, B.data,
                       np.ascontiguousarray(ia, np.int64), np.ascontiguousarray(ib, np.int64), out)
            return out
        except Exception as e:
            log('numba rowdot failed, scipy fallback:', e)
    for s in range(0, len(ia), chunk):
        out[s:s + chunk] = np.asarray(A[ia[s:s + chunk]].multiply(B[ib[s:s + chunk]]).sum(axis=1)).ravel()
    return out


# ============================================================================ stage 2: blocking
BLOCK_COLS = ['entity_id', 'country', 'name_core', 'name_phon', 'domain', 'addr_full', 'addr_words', 'addr_nums', 'locality',
              'street']


def embed(texts_fit, texts, dim, seed=0):
    """hashed char TF-IDF (pruned) -> truncated SVD -> L2-normalised float32 rows."""
    from sklearn.decomposition import TruncatedSVD
    vec = CharTfidf(prune=True, min_df=2 if len(texts_fit) > 1000 else 1).fit(texts_fit, n=200_000)
    rng = np.random.default_rng(seed)
    samp = [texts_fit[i] for i in rng.choice(len(texts_fit), min(len(texts_fit), 200_000), replace=False)]
    n_comp = min(dim, vec.dim - 1, len(samp) - 1)
    if n_comp < 1:                                           # tiny group (e.g. a handful of malformed rows)
        return [np.zeros((len(arr), 1), np.float32) for arr in texts]
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm='randomized', n_iter=4)
    svd.fit(vec._one(samp))
    V = svd.components_.T.astype(np.float32)                 # (vocab, dim)
    outs = []
    use_gpu = n_gpus() > 0 and os.environ.get('ER_GPU_PROJECT', '1') == '1'
    if use_gpu:
        import torch
        Vg = torch.from_numpy(V).cuda()
    for arr in texts:
        E = np.empty((len(arr), V.shape[1]), np.float32)
        for s in range(0, len(arr), 1_000_000):
            X = vec.transform(arr[s:s + 1_000_000])
            if use_gpu:
              try:
                # embedding_bag = sparse(X) @ V as a weighted sum of V rows; mature and stable kernel
                # (torch's beta sparse-CSR matmul crashed the kernel on some CUDA/GPU combinations)
                idx = torch.from_numpy(X.indices.astype(np.int64)).cuda()
                off = torch.from_numpy(X.indptr[:-1].astype(np.int64)).cuda()
                w = torch.from_numpy(X.data.astype(np.float32)).cuda()
                Y = torch.nn.functional.embedding_bag(idx, Vg, off, mode='sum', per_sample_weights=w)
                Y = torch.nn.functional.normalize(Y, dim=1)
                E[s:s + len(Y)] = Y.cpu().numpy(); del idx, off, w, Y
                continue
              except Exception as e:
                log('    GPU projection failed, CPU fallback:', str(e)[:120]); use_gpu = False
            if True:
                Y = np.asarray(X @ V); nrm = np.linalg.norm(Y, axis=1, keepdims=True); nrm[nrm == 0] = 1
                E[s:s + len(Y)] = Y / nrm
        outs.append(E)
    return outs


_FAISS_NOTE = []


def _faiss_gpu_ok():
    """The PyPI faiss-gpu-cu12 wheel ships no kernels for Hopper (H100/H200, compute 9.x): calling it
    there aborts the whole process ('no kernel image is available'). Use FAISS-GPU only below 9.0."""
    try:
        import torch
        caps = [torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())]
    except Exception:
        return True
    ok = all(c[0] < 9 for c in caps)
    if not ok and not _FAISS_NOTE:
        _FAISS_NOTE.append(1)
        log(f'    GPU compute capability {caps[0]} (Hopper): faiss-gpu wheel unsupported -> torch top-K on GPU')
    return ok


def search(ea, eb, k, mode='exact'):
    """Top-K inner product. FAISS on all GPUs (exact flat or IVF), else torch, else numpy."""
    k = min(k, len(eb)); t = time.time()
    try:
        if os.environ.get('ER_NO_FAISS', '0') == '1':
            raise ImportError('disabled by ER_NO_FAISS')
        import faiss
        ng = faiss.get_num_gpus()
        have_faiss = True
        if ng > 0 and not _faiss_gpu_ok():
            ng, have_faiss = 0, False                # -> torch top-K on the GPU below
    except Exception:
        ng, have_faiss = 0, False
    if mode == 'auto':
        mode = 'exact' if ng > 0 else 'ivf'
    if have_faiss and ng == 0:                     # CPU-only machine (laptop / CPU server): FAISS on all cores
        d = eb.shape[1]
        faiss.omp_set_num_threads(N_JOBS)
        if mode == 'ivf' and len(eb) > 50_000:
            nlist = int(min(16384, max(256, 4 * np.sqrt(len(eb)))))
            q = faiss.IndexFlatIP(d)
            idx = faiss.IndexIVFFlat(q, d, nlist, faiss.METRIC_INNER_PRODUCT)
            idx.train(np.ascontiguousarray(eb[np.random.default_rng(0).choice(len(eb), min(len(eb), 40 * nlist), replace=False)]))
            idx.add(np.ascontiguousarray(eb)); idx.nprobe = 64
        else:
            idx = faiss.IndexFlatIP(d); idx.add(np.ascontiguousarray(eb))
        I = np.empty((len(ea), k), np.int64); S = np.empty((len(ea), k), np.float32)
        for s in range(0, len(ea), 65536):
            D, J = idx.search(np.ascontiguousarray(ea[s:s + 65536]), k)
            S[s:s + 65536], I[s:s + 65536] = D, J
        log(f'    faiss-cpu-{mode}: {len(ea):,} x {len(eb):,} in {time.time() - t:.0f}s')
        return I, S
    if ng > 0:
      try:
        d = eb.shape[1]
        if mode == 'ivf':
            nlist = int(min(16384, max(256, 4 * np.sqrt(len(eb)))))
            cpu = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, nlist, faiss.METRIC_INNER_PRODUCT)
            co = faiss.GpuMultipleClonerOptions(); co.shard = True; co.useFloat16 = True
            idx = faiss.index_cpu_to_all_gpus(cpu, co=co)
            tr = eb[np.random.default_rng(0).choice(len(eb), min(len(eb), 40 * nlist), replace=False)]
            idx.train(np.ascontiguousarray(tr)); idx.add(np.ascontiguousarray(eb))
            faiss.GpuParameterSpace().set_index_parameter(idx, 'nprobe', 64)
        else:
            co = faiss.GpuMultipleClonerOptions(); co.shard = True; co.useFloat16 = True
            idx = faiss.index_cpu_to_all_gpus(faiss.IndexFlatIP(d), co=co)
            idx.add(np.ascontiguousarray(eb))
        I = np.empty((len(ea), k), np.int64); S = np.empty((len(ea), k), np.float32)
        for s in range(0, len(ea), 131072):
            D, J = idx.search(np.ascontiguousarray(ea[s:s + 131072]), k)
            S[s:s + 131072], I[s:s + 131072] = D, J
        del idx
        log(f'    faiss-{mode} x{ng}: {len(ea):,} x {len(eb):,} in {time.time() - t:.0f}s')
        return I, S
      except Exception as e:   # e.g. a faiss wheel without kernels for this GPU architecture
        log('    faiss failed -> torch top-K:', str(e)[:150]); t = time.time()
    if n_gpus() > 0:
        import torch
        nb = len(eb); BLK = 1 << 16
        nbp = -(-nb // BLK) * BLK if nb > 4 * BLK else nb
        B = torch.zeros((nbp, eb.shape[1]), dtype=torch.float16, device='cuda'); B[:nb] = torch.from_numpy(eb).to('cuda', torch.float16)
        rows = int(max(64, min(8192, 0.30 * torch.cuda.mem_get_info()[0] // (2 * nbp))))
        I = np.empty((len(ea), k), np.int64); S = np.empty((len(ea), k), np.float32)

        def plain(M):
            return torch.topk(M, k, dim=1)

        def blockwise(M):   # torch.topk on a 6M-wide row runs one CTA per row; 64k-wide blocks parallelise
            r = M.shape[0]
            v1, i1 = torch.topk(M.view(r, nbp // BLK, BLK), k, dim=2)
            v2, j = torch.topk(v1.reshape(r, -1), k, dim=1)
            return v2, (j // k) * BLK + i1.reshape(r, -1).gather(1, j)
        fn = blockwise if nbp != nb else plain
        for s in range(0, len(ea), rows):
            M = torch.from_numpy(ea[s:s + rows]).to('cuda', torch.float16) @ B.T
            if nbp != nb:
                M[:, nb:] = -60000.0
            if s == 0 and fn is blockwise:          # self-check against the plain kernel, else fall back
                try:
                    va, _ = blockwise(M[:32]); vb, _ = plain(M[:32])
                    if not torch.allclose(va.float(), vb.float(), atol=1e-3):
                        raise ValueError('mismatch')
                except Exception as e:
                    log('    blockwise top-K disabled:', str(e)[:80]); fn = plain
            v, i = fn(M); del M
            I[s:s + rows] = i.cpu().numpy(); S[s:s + rows] = v.float().cpu().numpy()
        del B; torch.cuda.empty_cache()
        log(f'    torch top-K{"(blockwise)" if fn is blockwise else ""}: {len(ea):,} x {len(eb):,} (k={k}) in {time.time() - t:.0f}s')
        return I, S
    log('    WARNING: no GPU -> numpy top-K (slow)')
    return topk_dense(ea, eb, k)


UNION_NAMES = ['name', 'addr', 'both', 'key', 'rev', 'hop']


def union_candidates(parts):
    """parts: (pass id, i1, i2, rank, sim). Pass ids 4 (reverse search, all views) and 5 (2-hop) can
    hold the same pair several times -> max sim / min rank via ufunc.at."""
    i1 = np.concatenate([p[1] for p in parts]).astype(np.int64)
    i2 = np.concatenate([p[2] for p in parts]).astype(np.int64)
    uc, inv = np.unique(i1 << 32 | i2, return_inverse=True)
    del i1, i2
    names = UNION_NAMES
    C = {'i1': (uc >> 32).astype(np.int32), 'i2': (uc & 0xFFFFFFFF).astype(np.int32)}
    sims = {n: np.full(len(uc), -1, np.float32) for n in names}
    ranks = {n: np.full(len(uc), np.inf, np.float32) for n in names}
    off = 0
    for pid, a, b, r, s in parts:
        idx = inv[off:off + len(a)]; off += len(a); n = names[pid]
        if pid >= 4:
            np.maximum.at(sims[n], idx, s.astype(np.float32)); np.minimum.at(ranks[n], idx, r.astype(np.float32))
        else:
            sims[n][idx] = np.maximum(sims[n][idx], s); ranks[n][idx] = np.minimum(ranks[n][idx], r)
    for n in names:
        ranks[n][np.isinf(ranks[n])] = -1
    for n in ['addr', 'both', 'key', 'name', 'rev', 'hop']:
        C['sim_' + n] = sims[n]
    for n in ['addr', 'both', 'key', 'name', 'rev', 'hop']:
        C['rank_' + n] = ranks[n]
    C['n_passes'] = sum((sims[n] > -1).astype(np.int8) for n in names).astype(np.int8)
    return pd.DataFrame(C)


def stage_block(cfg, split, countries):
    for c in countries:
        wc = f"{cfg['work']}/{split}/{c}"
        bfp = {'ks': cfg['ks'], 'key_k': cfg['key_k'], 'dim': cfg['dim'], 'search': cfg.get('search', 'exact'),
               'rev_k': cfg.get('rev_k', 3), 'hop_k': cfg.get('hop_k', 3), 'v': 3}
        old = json.load(open(f'{wc}/block_fp.json')) if os.path.exists(f'{wc}/block_fp.json') else None
        if old != bfp and (os.path.exists(f'{wc}/C.pkl') or glob.glob(f'{wc}/view_*.npz')):
            import shutil
            log(split, c, 'blocking settings changed -> recomputing stage 2 and everything after it')
            for f in glob.glob(f'{wc}/view_*.npz') + glob.glob(f'{wc}/emb2.npy') + [f'{wc}/C.pkl', f'{wc}/pred.pkl', f'{wc}/pA_train.npy']:
                if os.path.exists(f): os.remove(f)
            shutil.rmtree(f'{wc}/feats', ignore_errors=True); shutil.rmtree(f"{cfg['work']}/models", ignore_errors=True)
        json.dump(bfp, open(f'{wc}/block_fp.json', 'w'))
        if os.path.exists(f'{wc}/C.pkl') and os.path.exists(f'{wc}/emb2.npy'):
            log(split, c, 'block: cached'); continue
        E1 = ColStore.load(f'{wc}/E1', BLOCK_COLS); E2 = ColStore.load(f'{wc}/E2', BLOCK_COLS)
        if len(E1) == 0 or len(E2) == 0:
            pd.DataFrame({'i1': np.array([], np.int32), 'i2': np.array([], np.int32)}).to_pickle(f'{wc}/C.pkl')
            np.save(f'{wc}/emb2.npy', np.zeros((len(E2), 1), np.float16)); continue
        log(split, repr(c), f'blocking {len(E1):,} S1 x {len(E2):,} S2/S3'
            + ('   <- tiny group: probably malformed rows with an odd country value' if len(E1) < 100 else ''))
        pool = ThreadPoolExecutor(1)        # GPU search of view v overlaps CPU embedding of view v+1
        futs = {}; prev = None
        for pid, (view, k) in enumerate(cfg['ks'].items()):
            f = f'{wc}/view_{view}.npz'
            if os.path.exists(f) and (view != 'both' or os.path.exists(f'{wc}/emb2.npy')):
                continue
            ta = VIEWS[view](E1).tolist(); tb = VIEWS[view](E2).tolist()
            rs = np.random.default_rng(1)
            fit = [ta[i] for i in rs.choice(len(ta), min(len(ta), 100_000), replace=False)] + \
                  [tb[i] for i in rs.choice(len(tb), min(len(tb), 300_000), replace=False)]
            ea, eb = embed(fit, [ta, tb], cfg['dim']); del fit
            del ta, tb
            if view == 'both':                # kept for the neighbour-support features of stage 4
                np.save(f'{wc}/emb2.npy', eb.astype(np.float16))
            log(split, c, 'view', view, 'embedded')
            def job(ea=ea, eb=eb, k=k, f=f, view=view):
                mode = cfg.get('search', 'exact'); extra = {}
                I, S = search(ea, eb, k, mode)
                rk = int(cfg.get('rev_k', 3))
                if rk > 0:                    # reverse: each S2/S3 record's nearest S1 records (one owner each)
                    Ir, Sr = search(eb, ea, rk, mode); extra.update(Ir=Ir.astype(np.int32), Sr=Sr.astype(np.float16))
                hk = int(cfg.get('hop_k', 3))
                if hk > 0 and view == 'name':   # 2-hop: S2/S3 neighbours of S2/S3 records (siblings of a match)
                    H, HS = search(eb, eb, hk + 1, mode); extra.update(H=H.astype(np.int32), HS=HS.astype(np.float16))
                np.savez(f, I=I.astype(np.int32), S=S.astype(np.float32), **extra)
                log(split, c, 'view', view, 'searched')
            if prev is not None:
                prev.result()                # keep at most 2 views' embeddings in RAM
            prev = futs[view] = pool.submit(job)
            del ea, eb; gc.collect()
        kf = f'{wc}/view_key.npz'
        if not os.path.exists(kf):
            K = key_candidates_np(E1, E2, k=cfg['key_k'], max_df=40)
            np.savez(kf, i1=K.i1.values, i2=K.i2.values, r=K['rank'].values.astype(np.float32), s=K.sim.values)
            log(split, c, 'rare-key pass done', len(K)); del K
        for v, fu in futs.items():
            fu.result()                      # re-raises errors from the search thread
        pool.shutdown()
        parts = []; n1, n2 = len(E1), len(E2)
        seeds = []; H = None
        for pid, view in enumerate(cfg['ks']):
            z = np.load(f'{wc}/view_{view}.npz'); I = z['I']; kk = I.shape[1]
            parts.append((pid, np.repeat(np.arange(n1, dtype=np.int32), kk), I.ravel(),
                          np.tile(np.arange(kk, dtype=np.float32), n1), z['S'].ravel()))
            if 'Ir' in z.files:
                Ir = z['Ir']; rk = Ir.shape[1]
                parts.append((4, Ir.ravel(), np.repeat(np.arange(n2, dtype=np.int32), rk),
                              np.tile(np.arange(rk, dtype=np.float32), n2), z['Sr'].ravel().astype(np.float32)))
            if view in ('name', 'both'):
                seeds.append(I[:, :3])
            if 'H' in z.files:
                H, HS = z['H'], z['HS'].astype(np.float32)
        z = np.load(kf); parts.append((3, z['i1'], z['i2'], z['r'], z['s']))
        if int(cfg.get('hop_k', 3)) > 0 and H is not None:
            sd = np.concatenate(seeds, 1)                               # n1 x 6 seed records
            self_ = H == np.arange(n2)[:, None]                         # drop each record's self-hit
            hk = H.shape[1] - 1
            Hn = np.where(self_.any(1, keepdims=True), np.where(self_, -1, H), H)
            HSn = np.where(Hn < 0, -1, HS)
            nb = Hn[sd.ravel()].reshape(n1, -1); ns = HSn[sd.ravel()].reshape(n1, -1)
            rr = np.tile(np.arange(H.shape[1], dtype=np.float32), sd.shape[1])
            a1 = np.repeat(np.arange(n1, dtype=np.int32), nb.shape[1]); m = nb.ravel() >= 0
            parts.append((5, a1[m], nb.ravel()[m], np.tile(rr, n1)[m], ns.ravel()[m]))
            del sd, Hn, HSn, nb, ns, a1, m
            log(split, c, f'2-hop expansion (k={hk}) added')
        C = union_candidates(parts); C.to_pickle(f'{wc}/C.pkl')
        log(split, c, 'candidates', f'{len(C):,}', 'per S1 %.1f' % (len(C) / len(E1)))
        del parts, C, E1, E2; gc.collect()


# ============================================================================ stage 3: features
class Pairs:
    """Pairs sorted by i2, cut into E2 row blocks -> bounded memory for E2-side matrices."""
    def __init__(self, i1, i2, n2, blk=1_500_000):
        self.i1, self.i2 = i1, i2
        self.order = np.argsort(i2, kind='stable')
        s2 = i2[self.order]
        self.blocks = []
        for s in range(0, n2, blk):
            lo, hi = np.searchsorted(s2, [s, s + blk])
            if hi > lo:
                self.blocks.append((s, min(s + blk, n2), lo, hi))


def _cos(vec, t1, t2, P):
    A = vec.transform(t1)
    out = np.empty(len(P.i1), np.float32)
    for s, e, lo, hi in P.blocks:
        B = vec.transform(t2[s:e]); idx = P.order[lo:hi]
        out[idx] = rowdot(A, B, P.i1[idx], P.i2[idx] - s)
    return out


def _overlap(t1, t2, P):
    A = word_counts(t1); nA = np.diff(A.indptr).astype(np.float32)
    inter = np.empty(len(P.i1), np.float32); n2 = np.empty(len(P.i1), np.float32)
    for s, e, lo, hi in P.blocks:
        B = word_counts(t2[s:e]); idx = P.order[lo:hi]
        inter[idx] = rowdot(A, B, P.i1[idx], P.i2[idx] - s)
        n2[idx] = np.diff(B.indptr)[P.i2[idx] - s]
    n1 = nA[P.i1]; union = n1 + n2 - inter; mn = np.minimum(n1, n2)
    jac = np.where(union > 0, inter / np.maximum(union, 1), -1).astype(np.float32)
    cont = np.where(mn > 0, inter / np.maximum(mn, 1), -1).astype(np.float32)
    return jac, cont, inter


def lev_pairs(a, b):
    try:
        from rapidfuzz.process import cpdist
        from rapidfuzz.distance import Levenshtein
        return cpdist(list(a), list(b), scorer=Levenshtein.distance, workers=-1).astype(np.float32)
    except ImportError:
        return np.array([_lev(x, y) for x, y in zip(a, b)], np.float32)


def _lev(a, b):
    if a == b:
        return 0
    if not a or not b:
        return max(len(a), len(b))
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def pair_features(wc, C, out_dir, chunk=5_000_000):
    """One float16 .npy per feature; an existing file is skipped (resume per feature)."""
    os.makedirs(out_dir, exist_ok=True)
    i1, i2 = C.i1.values.astype(np.int64), C.i2.values.astype(np.int64)
    n2 = ColStore.n(f'{wc}/E2')
    P = Pairs(i1, i2, n2)
    names = []
    have = lambda *ks: all(os.path.exists(f'{out_dir}/{k}.npy') for k in ks)
    def put(k, v):
        np.save(f'{out_dir}/{k}.npy', np.asarray(v, dtype=np.float16))
    def col(side, c):
        return pd.read_pickle(f'{wc}/{side}/{c}.pkl').tolist()
    def ocol(side, c):
        return pd.read_pickle(f'{wc}/{side}/{c}.pkl').to_numpy(dtype=object)
    def group(keys, fn):
        names.extend(keys)
        if have(*keys):
            return
        t = time.time()
        vals = fn()
        vals = vals if isinstance(vals, tuple) else (vals,)
        for k, v in zip(keys, vals):
            put(k, v)
        log(f'    {",".join(keys)} ({time.time() - t:.0f}s)')
        gc.collect()
    ch = lambda **kw: (lambda c1, c2: _cos(CharTfidf(**kw).fit(col('E1', c1) + col('E2', c2)), col('E1', c1), col('E2', c2), P))
    # --- name ---
    group(['n_char'], lambda: ch()('name_core', 'name_core'))
    group(['n_full_char'], lambda: ch()('name_full', 'name_full'))
    group(['n_phon'], lambda: ch(ns=(1, 2, 3))('name_phon', 'name_phon'))
    group(['n_word'], lambda: _cos(WordTfidf(1).fit(col('E1', 'name_core') + col('E2', 'name_core')),
                                   col('E1', 'name_core'), col('E2', 'name_core'), P))
    group(['n_sorted'], lambda: ch(ns=(3,))('name_sorted', 'name_sorted'))
    group(['n_jac', 'n_cont', 'n_inter'], lambda: _overlap(col('E1', 'name_core'), col('E2', 'name_core'), P))
    group(['p_jac', 'p_cont'], lambda: _overlap(col('E1', 'name_phon'), col('E2', 'name_phon'), P)[:2])
    def concat():
        dom, nc = col('E2', 'domain'), col('E2', 'name_concat')
        d2 = [d if d else x for d, x in zip(dom, nc)]
        v = CharTfidf(ns=(3,), wb=False).fit(col('E1', 'name_concat') + d2)
        return _cos(v, col('E1', 'name_concat'), d2, P)
    group(['n_concat'], concat)
    def dom_in():
        dom, conc = ocol('E2', 'domain'), ocol('E1', 'name_concat')
        v = np.full(len(i1), -1, np.float32); h = np.flatnonzero(dom[i2] != '')
        v[h] = [float(d in c) for d, c in zip(dom[i2[h]], conc[i1[h]])]
        return v
    group(['dom_in_name'], dom_in)
    def first_eq():
        f1, f2 = ocol('E1', 'first_tok'), ocol('E2', 'first_tok')
        return np.concatenate([(f1[i1[s:s + chunk]] == f2[i2[s:s + chunk]]) & (f1[i1[s:s + chunk]] != '')
                               for s in range(0, len(i1), chunk)])
    group(['n_first_eq'], first_eq)
    def fuzzy():
        try:
            from rapidfuzz.process import cpdist
            from rapidfuzz import fuzz
            from rapidfuzz.distance import JaroWinkler
        except ImportError:
            z = np.full(len(i1), -1, np.float32); return z, z, z
        nf1, nf2 = ocol('E1', 'name_full'), ocol('E2', 'name_full')
        nc1, nc2 = ocol('E1', 'name_core'), ocol('E2', 'name_core')
        r = {k: [] for k in 'tpj'}
        for s_ in range(0, len(i1), chunk):
            a, b = i1[s_:s_ + chunk], i2[s_:s_ + chunk]
            r['t'].append(cpdist(list(nf1[a]), list(nf2[b]), scorer=fuzz.token_set_ratio, workers=-1) / 100)
            r['p'].append(cpdist(list(nc1[a]), list(nc2[b]), scorer=fuzz.partial_ratio, workers=-1) / 100)
            r['j'].append(cpdist(list(nc1[a]), list(nc2[b]), scorer=JaroWinkler.similarity, workers=-1))
        return tuple(np.concatenate(r[k]).astype(np.float32) for k in 'tpj')
    group(['rf_token_set', 'rf_partial', 'rf_jaro'], fuzzy)
    # --- address ---
    group(['a_char'], lambda: ch()('addr_full', 'addr_full'))
    group(['a_word'], lambda: _cos(WordTfidf(2).fit(col('E1', 'addr_words') + col('E2', 'addr_words')),
                                   col('E1', 'addr_words'), col('E2', 'addr_words'), P))
    group(['a_jac', 'a_cont'], lambda: _overlap(col('E1', 'addr_words'), col('E2', 'addr_words'), P)[:2])
    group(['num_jac', 'num_cont', 'num_inter'], lambda: _overlap(col('E1', 'addr_nums'), col('E2', 'addr_nums'), P))
    def house():
        H1, H2 = ocol('E1', 'house_no'), ocol('E2', 'house_no')
        D1, D2 = ocol('E1', 'house_dig'), ocol('E2', 'house_dig')
        T1, T2 = ocol('E1', 'num_tok'), ocol('E2', 'num_tok')
        int1 = np.array([int(x[:9]) if x else -1 for x in D1], np.float64)
        int2 = np.array([int(x[:9]) if x else -1 for x in D2], np.float64)
        len1 = np.array([len(x) for x in D1], np.float32); len2 = np.array([len(x) for x in D2], np.float32)
        suf1 = np.array([x[-2:] for x in D1], object); suf2 = np.array([x[-2:] for x in D2], object)
        cols = {k: [] for k in ['house_eq', 'house_dig_eq', 'house_suffix_eq', 'house_len_diff', 'house_lev', 'house_logdiff', 'num_tok_lev']}
        for s in range(0, len(i1), chunk):
            a, b = i1[s:s + chunk], i2[s:s + chunk]
            h1, h2 = H1[a], H2[b]
            cols['house_eq'].append(np.where((h1 == '') | (h2 == ''), -1, (h1 == h2).astype(np.int8)))
            d1, d2 = D1[a], D2[b]; both = (d1 != '') & (d2 != '')
            cols['house_dig_eq'].append(np.where(both, (d1 == d2).astype(np.int8), -1))
            cols['house_suffix_eq'].append(np.where(both, (suf1[a] == suf2[b]).astype(np.int8), -1))
            cols['house_len_diff'].append(np.where(both, np.abs(len1[a] - len2[b]), -1))
            lv = np.full(len(a), -1, np.float32); w_ = np.flatnonzero(both); lv[w_] = lev_pairs(d1[w_], d2[w_])
            cols['house_lev'].append(lv)
            cols['house_logdiff'].append(np.where(both, np.log1p(np.abs(int1[a] - int2[b])), -1))
            t1, t2 = T1[a], T2[b]
            lv = np.full(len(a), -1, np.float32); w_ = np.flatnonzero((t1 != '') & (t2 != '')); lv[w_] = lev_pairs(t1[w_], t2[w_])
            cols['num_tok_lev'].append(lv)
        return tuple(np.concatenate(cols[k]) for k in cols)
    group(['house_eq', 'house_dig_eq', 'house_suffix_eq', 'house_len_diff', 'house_lev', 'house_logdiff', 'num_tok_lev'], house)
    group(['addr_empty2'], lambda: pd.read_pickle(f'{wc}/E2/addr_empty.pkl').values[i2])
    group(['street_char'], lambda: ch()('street', 'street'))
    group(['street_jac', 'street_cont'], lambda: _overlap(col('E1', 'street'), col('E2', 'street'), P)[:2])
    group(['loc_jac'], lambda: _overlap(col('E1', 'locality'), col('E2', 'locality'), P)[0])
    group(['n_distinct_char'], lambda: ch()('name_distinct', 'name_distinct'))
    group(['n_distinct_jac'], lambda: _overlap(col('E1', 'name_distinct'), col('E2', 'name_distinct'), P)[0])
    def legal():
        jac, _, li = _overlap(col('E1', 'legal'), col('E2', 'legal'), P)
        L1, L2 = ocol('E1', 'legal'), ocol('E2', 'legal')
        e1, e2 = (L1 == '')[i1], (L2 == '')[i2]
        eq = np.concatenate([np.where(e1[s:s + chunk] | e2[s:s + chunk], -1,
                                      (L1[i1[s:s + chunk]] == L2[i2[s:s + chunk]]).astype(np.int8)) for s in range(0, len(i1), chunk)])
        return jac, eq, (~e1) & (~e2) & (li == 0), e1, e2
    group(['lf_jac', 'lf_eq', 'lf_conflict', 'lf_empty1', 'lf_empty2'], legal)
    def flags():
        g = lambda side, c: pd.read_pickle(f'{wc}/{side}/{c}.pkl').values
        ln1, ln2 = g('E1', 'n_len').astype(np.float32), g('E2', 'n_len').astype(np.float32)
        return (g('E2', 'name_native')[i2], g('E2', 'addr_native')[i2], g('E2', 'is_domain')[i2],
                (ocol('E2', 'src') == 'S3')[i2], ln1[i1], ln2[i2], np.abs(ln1[i1] - ln2[i2]))
    group(['native2', 'addr_native2', 'is_domain2', 'is_s3', 'n_len1', 'n_len2', 'n_len_diff'], flags)
    bl = [c for c in ['sim_addr', 'sim_both', 'sim_key', 'sim_name', 'sim_rev', 'sim_hop', 'rank_addr', 'rank_both',
                      'rank_key', 'rank_name', 'rank_rev', 'rank_hop', 'n_passes'] if c in C.columns]
    group(bl, lambda: tuple(C[c].values for c in bl))
    ld = lambda k: np.load(f'{out_dir}/{k}.npy').astype(np.float32)
    def combo():
        nb = np.maximum.reduce([ld('n_char'), ld('n_phon'), ld('n_concat')])
        return nb, nb * np.where(ld('addr_empty2') == 1, 0.5, ld('a_char'))
    group(['name_best', 'name_x_addr'], combo)
    json.dump(names, open(f'{out_dir}/_names.json', 'w'))
    return names


def stage_features(cfg, split, countries):
    for c in countries:
        wc = f"{cfg['work']}/{split}/{c}"
        if os.path.exists(f'{wc}/feats/_names.json'):
            log(split, c, 'features: cached'); continue
        if not os.path.exists(f'{wc}/C.pkl'):
            continue
        C = pd.read_pickle(f'{wc}/C.pkl')
        if len(C) == 0:
            continue
        log(split, c, f'features for {len(C):,} pairs')
        pair_features(wc, C, f'{wc}/feats')
        log(split, c, 'features done')
        del C; gc.collect()


# ============================================================================ stage 4: model
def competition_features(i1, i2, s):
    """Graph context from a pair score, numpy sort-based (no pandas groupby)."""
    n = len(s); out = {}
    for side, ids in (('1', i1), ('2', i2)):
        o = np.lexsort((-s, ids)); ss = s[o]; gi = ids[o]
        start = np.r_[0, np.flatnonzero(np.diff(gi)) + 1]
        size = np.diff(np.r_[start, n])
        grp = np.repeat(np.arange(len(start)), size)
        rank = (np.arange(n) - start[grp] + 1).astype(np.float32)
        top = ss[start][grp]
        r = np.empty(n, np.float32)
        if side == '1':
            r[o] = rank; out['a_rank1'] = r.copy()
            r[o] = top - ss; out['a_gap1'] = r.copy()
            close = np.add.reduceat((ss > top - 0.1).astype(np.float32), start)[grp]
            r[o] = close; out['a_n_close1'] = r.copy()
            r[o] = np.add.reduceat(ss, start)[grp]; out['a_sum1'] = r.copy()
        else:
            r[o] = rank; out['a_rank2'] = r.copy()
            second = np.where(size > 1, ss[np.minimum(start + 1, n - 1)], -1.0)[grp]
            other = np.where(rank == 1, second, top)
            r[o] = ss - other; out['a_margin2'] = r.copy()
            r[o] = size[grp]; out['a_n_owner2'] = r.copy()
    return {k: out[k] for k in ['a_rank1', 'a_gap1', 'a_n_close1', 'a_sum1', 'a_rank2', 'a_margin2', 'a_n_owner2']}


def support_features(i1, i2, p, emb, M=5, chunk=2_000_000):
    """S2/S3 hold several records of the same business. For pair (a, b): take the M most likely
    candidates b' of a (by stage-A score) and measure how similar b is to them. A record that looks
    weak against the S1 name but is a near-copy of a confident match is probably a match too.
      s_max  = max_m p(a,b'_m) * cos(b, b'_m)      s_cnt = #{m : p>0.5 and cos>0.8}
      s_top  = cos(b, best b')                     (b' != b; emb = L2-normalised 'both' view)"""
    n = len(p)
    out = {'s_max': np.full(n, -1, np.float32), 's_cnt': np.zeros(n, np.float32), 's_top': np.full(n, -1, np.float32)}
    if n == 0 or emb.shape[1] < 2:
        return out
    o = np.lexsort((-p, i1)); si1 = i1[o]
    start = np.r_[0, np.flatnonzero(np.diff(si1)) + 1]; size = np.diff(np.r_[start, n])
    grp = np.repeat(np.arange(len(start)), size); pos = np.arange(n) - start[grp]
    Ai = np.full((len(start), M), -1, np.int64); Ap = np.zeros((len(start), M), np.float32)
    sel = pos < M
    Ai[grp[sel], pos[sel]] = i2[o][sel]; Ap[grp[sel], pos[sel]] = p[o][sel]
    gp = np.empty(n, np.int64); gp[o] = grp
    for s in range(0, n, chunk):
        b = i2[s:s + chunk]; g = gp[s:s + chunk]
        e = np.asarray(emb[b], np.float32)
        smax = np.full(len(b), -1, np.float32); scnt = np.zeros(len(b), np.float32); stop = np.full(len(b), -1, np.float32)
        for m in range(M):
            a = Ai[g, m]; ok = (a >= 0) & (a != b)
            cos = np.einsum('ij,ij->i', e, np.asarray(emb[np.where(ok, a, 0)], np.float32))
            cos = np.where(ok, cos, -1)
            w = np.where(ok, Ap[g, m] * cos, -1)
            smax = np.maximum(smax, w)
            scnt += ok & (Ap[g, m] > 0.5) & (cos > 0.8)
            stop = np.where((stop == -1) & ok, cos, stop)
        out['s_max'][s:s + chunk], out['s_cnt'][s:s + chunk], out['s_top'][s:s + chunk] = smax, scnt, stop
    return out


def graph_features(p_):
    """competition (one-owner) + neighbour-support features from the stage-A score p_['pA']."""
    i1, i2 = p_['C'].i1.values.astype(np.int64), p_['C'].i2.values.astype(np.int64)
    G = competition_features(i1, i2, p_['pA'])
    emb = np.load(f"{p_['wc']}/emb2.npy", mmap_mode='r') if os.path.exists(f"{p_['wc']}/emb2.npy") else np.zeros((1, 1), np.float16)
    G.update(support_features(i1, i2, p_['pA'], emb))
    return G


class Model:
    """XGBoost (GPU) with a CPU fallback. predict() returns P(match)."""
    def __init__(self, seed, gpu=None):
        self.seed, self.gpu, self.bst, self.sk = seed, gpu, None, None

    def fit(self, X, y, w=None):
        w = np.ones(len(y), np.float32) if w is None else w
        try:
            import xgboost as xgb
            dev = f'cuda:{self.gpu}' if self.gpu is not None else 'cpu'
            r = np.random.default_rng(self.seed).random(len(y)) < 0.05
            dtr = xgb.QuantileDMatrix(X[~r], y[~r], weight=w[~r], max_bin=256)
            dva = xgb.QuantileDMatrix(X[r], y[r], weight=w[r], ref=dtr)
            p = dict(objective='binary:logistic', eval_metric='logloss', tree_method='hist', device=dev,
                     max_depth=0, max_leaves=127, grow_policy='lossguide', eta=0.05, subsample=0.8,
                     colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0, max_bin=256, seed=self.seed,
                     nthread=N_JOBS)
            self.bst = xgb.train(p, dtr, num_boost_round=3000, evals=[(dva, 'val')],
                                 early_stopping_rounds=50, verbose_eval=250)
            self.best = self.bst.best_iteration + 1
            return self
        except ImportError:
            from sklearn.ensemble import HistGradientBoostingClassifier
            self.sk = HistGradientBoostingClassifier(max_iter=800, learning_rate=0.08, max_leaf_nodes=63,
                                                     min_samples_leaf=40, l2_regularization=1.0, early_stopping=True,
                                                     validation_fraction=0.1, n_iter_no_change=30,
                                                     random_state=self.seed).fit(X, y, sample_weight=w)
            return self

    def predict(self, X, chunk=4_000_000):
        out = np.empty(len(X), np.float32)
        for s in range(0, len(X), chunk):
            x = np.asarray(X[s:s + chunk], np.float32)
            out[s:s + chunk] = (self.bst.inplace_predict(x, iteration_range=(0, self.best)) if self.bst is not None
                                else self.sk.predict_proba(x)[:, 1])
        return out


class Store:
    def __init__(self, cfg, split, countries):
        self.parts = []
        for c in countries:
            wc = f"{cfg['work']}/{split}/{c}"
            if not os.path.exists(f'{wc}/feats/_names.json'):
                continue
            names = json.load(open(f'{wc}/feats/_names.json'))
            self.parts.append(dict(country=c, wc=wc, names=names,
                                   cols=[np.load(f'{wc}/feats/{k}.npy', mmap_mode='r') for k in names],
                                   C=pd.read_pickle(f'{wc}/C.pkl')[['i1', 'i2']],
                                   ids1=np.load(f'{wc}/ids1.npy', allow_pickle=True),
                                   ids2=np.load(f'{wc}/ids2.npy', allow_pickle=True)))

    @staticmethod
    def X(p, idx, extra=None):
        cols = p['cols'] + (extra or [])
        out = np.empty((len(idx), len(cols)), np.float32)
        for j, c in enumerate(cols):
            out[:, j] = c[idx]
        return out


def stage_train(cfg, countries):
    wm = f"{cfg['work']}/models"; os.makedirs(wm, exist_ok=True)
    if os.path.exists(f'{wm}/final.pkl'):
        log('train: cached'); return pickle.load(open(f'{wm}/final.pkl', 'rb'))
    st = Store(cfg, 'train', countries)
    gt = read_tsv(f"{cfg['data']}/train/train_ground_truth.tsv")
    GP = pd.DataFrame([(a, b) for a, ms in zip(gt.source1_entity_id, gt.matched_entity_ids) for b in ms.split(',') if b],
                      columns=['s1', 'cand'])
    for p in st.parts:
        C = p['C']
        a = GP.s1.map(pd.Series(np.arange(len(p['ids1'])), index=p['ids1']))
        b = GP.cand.map(pd.Series(np.arange(len(p['ids2'])), index=p['ids2']))
        ok = a.notna() & b.notna()
        g = a[ok].astype(np.int64).values << 32 | b[ok].astype(np.int64).values
        p['y'] = np.isin(C.i1.values.astype(np.int64) << 32 | C.i2.values.astype(np.int64), g).astype(np.int8)
        bk = np.array([zlib.crc32(x.encode()) % 10 for x in p['ids1']], np.int8)[C.i1.values]
        p['val'] = bk >= 8; p['half'] = (bk >= 4).astype(np.int8)
        n_true = int(a.notna().sum())               # every true pair of this country's S1 entities
        rec = p['y'].sum() / max(1, n_true)
        log('train', repr(p['country']), f"pairs {len(C):,} positives {int(p['y'].sum()):,} of {n_true:,} true pairs"
            f" -> blocking recall {rec:.3f}")
        if n_true > 1000 and rec < 0.85:
            log('WARNING: blocking recall is far below the ~0.98 measured in development -> data or stage-2 cache '
                'is inconsistent; delete the work folder and re-run from stage 1.')
    del GP; gc.collect()
    total = sum(len(p['y']) for p in st.parts)
    # v6: keep (almost) every positive, subsample negatives, re-weight them -> calibrated probabilities
    cap = cfg['max_train_pairs']; npos = int(sum(p['y'].sum() for p in st.parts)); nneg = total - npos
    rpos = min(1.0, 0.4 * cap / max(1, npos)); rneg = min(1.0, max(0.01, (cap - rpos * npos) / max(1, nneg)))
    wneg = rpos / rneg
    log(f'training rows: positives kept {rpos:.0%}, negatives kept {rneg:.1%} (weight {wneg:.2f})')
    rng = np.random.default_rng(0)
    masks = {p['country']: rng.random(len(p['y'])) < np.where(p['y'] == 1, rpos, rneg) for p in st.parts}

    def gather(sel, extra=None):
        Xs, ys = [], []
        for p in st.parts:
            idx = np.flatnonzero(sel(p) & masks[p['country']])
            Xs.append(Store.X(p, idx, extra(p) if extra else None)); ys.append(p['y'][idx])
        y = np.concatenate(ys)
        return np.vstack(Xs), y, np.where(y == 1, 1.0, wneg).astype(np.float32)

    ng = n_gpus()
    A = [None, None]
    for h in (0, 1):
        if os.path.exists(f'{wm}/A{h}.pkl'):
            A[h] = pickle.load(open(f'{wm}/A{h}.pkl', 'rb'))

    def fitA(h):
        X, y, w = gather(lambda p: (~p['val']) & (p['half'] == h))
        log(f'stage A{h}: fitting on {len(y):,} rows' + (f' (GPU {h % ng})' if ng else ''))
        m = Model(h, gpu=(h % ng) if ng else None).fit(X, y, w)
        pickle.dump(m, open(f'{wm}/A{h}.pkl', 'wb')); log(f'stage A{h} done')
        return m
    todo = [h for h in (0, 1) if A[h] is None]
    if ng >= 2 and len(todo) == 2:
        with ThreadPoolExecutor(2) as ex:                  # one model per GPU, concurrently
            A = list(ex.map(fitA, (0, 1)))
    else:
        for h in todo:
            A[h] = fitA(h)
    for p in st.parts:
        f = f"{p['wc']}/pA_train.npy"
        if os.path.exists(f):
            p['pA'] = np.load(f)
        else:
            pa = np.empty(len(p['y']), np.float32)
            for h in (0, 1):
                o = np.flatnonzero((~p['val']) & (p['half'] != h))
                for s in range(0, len(o), 4_000_000):
                    pa[o[s:s + 4_000_000]] = A[h].predict(Store.X(p, o[s:s + 4_000_000]))
            v = np.flatnonzero(p['val'])
            for s in range(0, len(v), 4_000_000):
                x = Store.X(p, v[s:s + 4_000_000]); pa[v[s:s + 4_000_000]] = (A[0].predict(x) + A[1].predict(x)) / 2
            np.save(f, pa); p['pA'] = pa
        p['G'] = graph_features(p)
        log('stage A scores + graph features', p['country'])
    gnames = list(st.parts[0]['G'])
    extra = lambda p: [p['G'][k] for k in gnames] + [p['pA']]
    if os.path.exists(f'{wm}/B.pkl'):
        B = pickle.load(open(f'{wm}/B.pkl', 'rb'))
    else:
        X, y, w = gather(lambda p: ~p['val'], extra)
        log(f'stage B: fitting on {len(y):,} rows')
        B = Model(7, gpu=0 if ng else None).fit(X, y, w); del X, y, w; gc.collect()
        pickle.dump(B, open(f'{wm}/B.pkl', 'wb'))
    rows, o1, o2 = [], 0, 0
    for p in st.parts:
        v = np.flatnonzero(p['val'])
        pb = np.concatenate([B.predict(Store.X(p, v[s:s + 4_000_000], extra(p)))
                             for s in range(0, len(v), 4_000_000)]) if len(v) else np.zeros(0, np.float32)
        rows.append(pd.DataFrame({'i1': p['C'].i1.values[v].astype(np.int64) + o1,
                                  'i2': p['C'].i2.values[v].astype(np.int64) + o2, 'p': pb,
                                  'part': np.int16(len(rows)), 'l1': p['C'].i1.values[v], 'l2': p['C'].i2.values[v]}))
        o1 += len(p['ids1']); o2 += len(p['ids2'])
    VR = pd.concat(rows, ignore_index=True)
    Vx = exclusive(VR[['i1', 'i2', 'p']])
    ids1 = np.concatenate([p['ids1'] for p in st.parts]); ids2 = np.concatenate([p['ids2'] for p in st.parts])
    val_ids = set(x for x in ids1 if zlib.crc32(x.encode()) % 10 >= 8)
    truth = {k: set(v.split(',')) - {''} for k, v in zip(gt.source1_entity_id, gt.matched_entity_ids) if k in val_ids}
    pickle.dump({'rows': VR, 'ids1': ids1, 'ids2': ids2, 'truth': truth, 'parts': [p['wc'] for p in st.parts]},
                open(f'{wm}/val.pkl', 'wb'))       # used by the optional cross-encoder stage
    grid = [(macro_f05(to_sets(by_threshold(Vx, t), ids1, ids2), truth), float(t)) for t in np.arange(0.40, 0.91, 0.025)]
    best_f, best_t = max(grid)
    f_exp = macro_f05(to_sets(by_expected_f(Vx), ids1, ids2), truth)
    rule = 'expected_f' if f_exp > best_f else 'threshold'
    log('VALIDATION macro F0.5: threshold %.4f (t=%.3f) | expected-F %.4f -> %s' % (best_f, best_t, f_exp, rule))
    try:   # where the points are lost (per country, error type) -> guides the next iteration
        from metric import f_beta_entity
        pred = to_sets(by_expected_f(Vx) if rule == 'expected_f' else by_threshold(Vx, best_t), ids1, ids2)
        c_of = {x: p['country'] for p in st.parts for x in p['ids1']}
        rows_ = []
        for k_, tv in truth.items():
            pv = pred.get(k_, set()); tp = len(pv & tv)
            cat = ('singleton_merged' if not tv and pv else 'fp+fn' if pv - tv and tv - pv else 'fp_only' if pv - tv
                   else 'fn_only' if tv - pv else 'perfect')
            rows_.append((c_of.get(k_, '?'), cat, f_beta_entity(pv, tv)))
        D_ = pd.DataFrame(rows_, columns=['country', 'cat', 'f'])
        diag = {'by_country': D_.groupby('country').f.mean().round(4).to_dict(),
                'points_lost_by_error': (D_.groupby('cat').f.apply(lambda x: (1 - x).sum()) / len(D_) * 100).round(3).to_dict()}
        log('DIAGNOSTICS', json.dumps(diag))
    except Exception as e:
        diag = {'error': str(e)}
    dec = {'threshold': best_t, 'f_threshold': best_f, 'f_expected': f_exp, 'rule': rule, 'grid': grid, 'gnames': gnames,
           'diagnostics': diag}
    models = {'A': A, 'B': B, 'dec': dec}
    pickle.dump(models, open(f'{wm}/final.pkl', 'wb'))
    os.makedirs(cfg['out'], exist_ok=True)
    json.dump(dec, open(f"{cfg['out']}/validation.json", 'w'), indent=1)
    return models


# ============================================================================ stage 5: predict
def stage_predict(cfg, countries, models):
    st = Store(cfg, 'test', countries)
    A, B, dec = models['A'], models['B'], models['dec']
    cand, match = {}, {}
    for p in st.parts:
        f = f"{p['wc']}/pred.pkl"
        if os.path.exists(f):
            T = pd.read_pickle(f)
        else:
            C = p['C']; n = len(C); allr = np.arange(n); pa = np.empty(n, np.float32)
            for s in range(0, n, 4_000_000):
                x = Store.X(p, allr[s:s + 4_000_000]); pa[s:s + 4_000_000] = (A[0].predict(x) + A[1].predict(x)) / 2
            p['pA'] = pa; G = graph_features(p)
            ex = [G[k] for k in dec['gnames']] + [pa]
            pb = np.empty(n, np.float32)
            for s in range(0, n, 4_000_000):
                pb[s:s + 4_000_000] = B.predict(Store.X(p, allr[s:s + 4_000_000], ex))
            T = pd.DataFrame({'i1': C.i1.values, 'i2': C.i2.values, 'p': pb}); T.to_pickle(f)
        D = by_expected_f(exclusive(T)) if dec['rule'] == 'expected_f' else by_threshold(exclusive(T), dec['threshold'])
        for dct, R in ((cand, T), (match, D)):
            g = pd.DataFrame({'s1': p['ids1'][R.i1.values], 'c': p['ids2'][R.i2.values]})
            dct.update(g.groupby('s1', sort=False).c.agg(','.join).to_dict())
        nS1 = len(p['ids1'])
        log('test', p['country'], f'S1 {nS1:,}', 'matches/S1 %.2f' % (len(D) / max(1, nS1)),
            'empty %.1f%%' % (100 * (1 - D.i1.nunique() / max(1, nS1))))
    order = read_tsv(f"{cfg['data']}/test/test_source1.tsv", usecols=['entity_id']).entity_id.to_numpy(dtype=object)
    os.makedirs(cfg['out'], exist_ok=True)
    for fn, colname, dct in (('candidate_pairs.tsv', 'candidate_entity_ids', cand), ('matching_results.tsv', 'matched_entity_ids', match)):
        out = pd.DataFrame({'source1_entity_id': order})
        out[colname] = out.source1_entity_id.map(dct).fillna('')
        out.to_csv(f"{cfg['out']}/{fn}", sep='\t', index=False)
        log('wrote', fn, f'{len(out):,} rows')


# ============================================================================ reuse an older run
def import_previous(old_work, new_work, splits=('train', 'test')):
    """Re-use stage-1 frames (E1/E2.pkl) and finished stage-2 candidates (C.pkl) written by the
    first Kaggle notebook (kaggle_er), so a switch to this version does not redo that work."""
    for split in splits:
        ow, nw = f'{old_work}/{split}', f'{new_work}/{split}'
        if not os.path.exists(f'{ow}/countries.json') or os.path.exists(f'{nw}/countries.json'):
            continue
        countries = json.load(open(f'{ow}/countries.json'))
        for c in countries:
            for side in ('E1', 'E2'):
                if not ColStore.exists(f'{nw}/{c}/{side}') and os.path.exists(f'{ow}/{c}/{side}.pkl'):
                    df = pd.read_pickle(f'{ow}/{c}/{side}.pkl')
                    ColStore.save(f'{nw}/{c}/{side}', df)
                    np.save(f'{nw}/{c}/ids{side[1]}.npy', df.entity_id.to_numpy(dtype=object), allow_pickle=True)
                    del df; gc.collect()
            if os.path.exists(f'{ow}/{c}/C.pkl') and not os.path.exists(f'{nw}/{c}/C.pkl'):
                import shutil; shutil.copy(f'{ow}/{c}/C.pkl', f'{nw}/{c}/C.pkl')
                log(split, c, 'reused stage-2 candidates from the previous run')
        s1 = pd.read_pickle(f'{ow}/s1_ids.pkl')
        np.save(f'{nw}/s1_order.npy', s1.entity_id.to_numpy(dtype=object), allow_pickle=True)
        json.dump(countries, open(f'{nw}/countries.json', 'w'))
        log(split, 'reused stage-1 output of the previous run for', countries)


In [ ]:
%%writefile {SRC}/run_stage.py
"""Runs one pipeline stage in its own process, so a native crash or an out-of-memory kill
cannot take the Jupyter kernel down; the notebook reports the exit reason instead."""
import sys, os, json, faulthandler

if __name__ == '__main__':            # guard required: forkserver workers import this file
    faulthandler.enable()             # prints a native traceback on segfault
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    import er_fast as K
    cfg = json.load(open(sys.argv[1])); stage = sys.argv[2]
    ctr = lambda: json.load(open(f"{cfg['work']}/train/countries.json"))
    cte = lambda: json.load(open(f"{cfg['work']}/test/countries.json"))
    if stage == 'normalise':
        K.stage_normalise(cfg, 'train'); K.stage_normalise(cfg, 'test')
    elif stage == 'block':
        K.stage_block(cfg, 'train', ctr()); K.stage_block(cfg, 'test', cte())
    elif stage == 'features_train':
        if not os.path.exists(f"{cfg['work']}/models/final.pkl"):
            K.stage_features(cfg, 'train', ctr())
    elif stage == 'train':
        K.stage_train(cfg, ctr())
    elif stage == 'predict':
        K.stage_features(cfg, 'test', cte()); K.stage_predict(cfg, cte(), K.stage_train(cfg, ctr()))
    elif stage == 'ce':
        import ce
        ce.stage_ce(cfg, ctr(), cte())
    elif stage == 'selftest_crash':
        import signal; os.kill(os.getpid(), signal.SIGKILL)
    else:
        raise SystemExit('unknown stage ' + stage)


In [ ]:
%%writefile {SRC}/ce.py
"""
Optional stage 6: cross-encoder re-ranker (Ditto-style) for the ambiguous pairs.

A small pre-trained transformer reads both records as text ("name | address" [SEP] "name | address")
and is fine-tuned to say match / no-match. It is applied only where the GBDT is unsure
(lo <= p <= hi), and its score is blended with the GBDT score:  p = w*p_gbdt + (1-w)*p_ce.
w, the threshold and the decision rule are tuned on the held-out validation entities (which the
cross-encoder never trains on); if the blend does not beat the GBDT alone on validation, w = 1
and the output is identical to stage 5.

Model: intfloat/multilingual-e5-small (MIT licence, ~118M parameters, far below the 8B cap).
"""
import os, gc, json, time, pickle, zlib
import numpy as np
import pandas as pd

import er_fast as K
from decide import exclusive, by_threshold, by_expected_f, to_sets
from metric import macro_f05

MODEL_NAME = os.environ.get('ER_CE_MODEL', 'intfloat/multilingual-e5-small')
LO, HI = 0.02, 0.995          # only these GBDT scores are re-scored
MAXLEN = 96


def _texts(wc, side):
    n = pd.read_pickle(f'{wc}/{side}/name_full.pkl').to_numpy(dtype=object)
    a = pd.read_pickle(f'{wc}/{side}/addr_full.pkl').to_numpy(dtype=object)
    return np.array([f'{x} | {y}'[:160] for x, y in zip(n, a)], dtype=object)


class Scorer:
    """Wraps tokenizer + model. ER_CE_DUMMY=1 -> a stand-in (for CPU plumbing tests only)."""
    def __init__(self, path=None):
        self.dummy = os.environ.get('ER_CE_DUMMY') == '1'
        if self.dummy:
            return
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        self.torch = torch
        self.dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        src = path or MODEL_NAME
        self.tok = AutoTokenizer.from_pretrained(src)
        self.model = AutoModelForSequenceClassification.from_pretrained(src, num_labels=1).to(self.dev)

    def _enc(self, a, b):
        return self.tok(list(a), list(b), truncation=True, max_length=MAXLEN, padding=True, return_tensors='pt')

    def fit(self, A, B, y, epochs=1, bs=256, lr=3e-5, log_every=2000):
        if self.dummy:
            return self
        torch = self.torch
        opt = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=0.01)
        steps = epochs * int(np.ceil(len(y) / bs)); warm = max(1, steps // 20)
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * max(0.0, 1 - s / steps))
        lossf = torch.nn.BCEWithLogitsLoss()
        self.model.train(); step = 0; t = time.time()
        for ep in range(epochs):
            perm = np.random.default_rng(ep).permutation(len(y))
            for s in range(0, len(y), bs):
                idx = perm[s:s + bs]
                enc = {k: v.to(self.dev) for k, v in self._enc(A[idx], B[idx]).items()}
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=self.dev == 'cuda'):
                    logit = self.model(**enc).logits.squeeze(-1)
                loss = lossf(logit.float(), torch.tensor(y[idx], dtype=torch.float32, device=self.dev))
                loss.backward(); torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
                if step % log_every == 0:
                    K.log(f'    CE step {step}/{steps} loss {loss.item():.4f} ({time.time() - t:.0f}s)')
        self.model.eval()
        return self

    def predict(self, A, B, bs=1024):
        if self.dummy:     # plumbing test: a noisy copy of a string-overlap score
            return np.array([len(set(a.split()) & set(b.split())) / max(1, len(set(a.split()) | set(b.split())))
                             for a, b in zip(A, B)], np.float32)
        torch = self.torch
        out = np.empty(len(A), np.float32)
        with torch.no_grad():
            for s in range(0, len(A), bs):
                enc = {k: v.to(self.dev) for k, v in self._enc(A[s:s + bs], B[s:s + bs]).items()}
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=self.dev == 'cuda'):
                    out[s:s + bs] = torch.sigmoid(self.model(**enc).logits.squeeze(-1).float()).cpu().numpy()
        return out

    def save(self, d):
        if not self.dummy:
            os.makedirs(d, exist_ok=True); self.model.save_pretrained(d); self.tok.save_pretrained(d)


def _labels(cfg, wc, C):
    ids1 = np.load(f'{wc}/ids1.npy', allow_pickle=True); ids2 = np.load(f'{wc}/ids2.npy', allow_pickle=True)
    gt = K.read_tsv(f"{cfg['data']}/train/train_ground_truth.tsv")
    GP = pd.DataFrame([(a, b) for a, ms in zip(gt.source1_entity_id, gt.matched_entity_ids) for b in ms.split(',') if b],
                      columns=['s1', 'cand'])
    a = GP.s1.map(pd.Series(np.arange(len(ids1)), index=ids1)); b = GP.cand.map(pd.Series(np.arange(len(ids2)), index=ids2))
    ok = a.notna() & b.notna()
    g = a[ok].astype(np.int64).values << 32 | b[ok].astype(np.int64).values
    y = np.isin(C.i1.values.astype(np.int64) << 32 | C.i2.values.astype(np.int64), g).astype(np.int8)
    val = np.array([zlib.crc32(x.encode()) % 10 >= 8 for x in ids1])[C.i1.values]
    return y, val


def stage_ce(cfg, countries_train, countries_test):
    wm = f"{cfg['work']}/models"
    if not os.path.exists(f'{wm}/final.pkl'):
        raise RuntimeError('run stage 4 first')
    models = pickle.load(open(f'{wm}/final.pkl', 'rb'))
    ce_dir = f'{wm}/ce'
    # ---------------- 1. fine-tune on ambiguous TRAIN pairs (never on validation entities) ----------
    if os.path.exists(f'{ce_dir}/done') or os.environ.get('ER_CE_DUMMY') == '1':
        sc = Scorer(ce_dir if os.path.exists(f'{ce_dir}/done') else None)
    else:
        A, B, Y = [], [], []
        for c in countries_train:
            wc = f"{cfg['work']}/train/{c}"
            if not os.path.exists(f'{wc}/pA_train.npy'):
                continue
            C = pd.read_pickle(f'{wc}/C.pkl'); pa = np.load(f'{wc}/pA_train.npy')
            y, val = _labels(cfg, wc, C)
            sel = np.flatnonzero((~val) & (pa > LO / 2))
            T1, T2 = _texts(wc, 'E1'), _texts(wc, 'E2')
            A.append(T1[C.i1.values[sel]]); B.append(T2[C.i2.values[sel]]); Y.append(y[sel])
            K.log('CE train rows', c, len(sel), 'positives', int(y[sel].sum()))
        A, B, Y = np.concatenate(A), np.concatenate(B), np.concatenate(Y)
        cap = int(cfg.get('ce_train_pairs', 3_000_000))
        if len(Y) > cap:
            k = np.random.default_rng(0).choice(len(Y), cap, replace=False); A, B, Y = A[k], B[k], Y[k]
        K.log(f'CE fine-tuning {MODEL_NAME} on {len(Y):,} pairs ({Y.mean():.1%} positive)')
        sc = Scorer().fit(A, B, Y.astype(np.float32))
        sc.save(ce_dir); open(f'{ce_dir}/done', 'w').close()
        del A, B, Y; gc.collect()
    # ---------------- 2. tune the blend on validation -------------------------------------------------
    V = pickle.load(open(f'{wm}/val.pkl', 'rb'))
    R = V['rows']; hard = np.flatnonzero((R.p.values >= LO) & (R.p.values <= HI))
    pce = R.p.values.copy()
    for part, wc in enumerate(V['parts']):
        m = hard[R.part.values[hard] == part]
        if len(m):
            T1, T2 = _texts(wc, 'E1'), _texts(wc, 'E2')
            pce[m] = sc.predict(T1[R.l1.values[m]], T2[R.l2.values[m]])
    K.log(f'CE scored {len(hard):,} ambiguous validation pairs')
    best = (-1, None)
    for w in (1.0, 0.75, 0.5, 0.25, 0.0):
        D = R[['i1', 'i2']].copy(); D['p'] = w * R.p.values + (1 - w) * pce
        X = exclusive(D)
        for t in np.arange(0.40, 0.91, 0.05):
            f = macro_f05(to_sets(by_threshold(X, t), V['ids1'], V['ids2']), V['truth'])
            if f > best[0]:
                best = (f, ('threshold', w, float(t)))
        f = macro_f05(to_sets(by_expected_f(X), V['ids1'], V['ids2']), V['truth'])
        if f > best[0]:
            best = (f, ('expected_f', w, None))
        K.log(f'  blend w={w:.2f}: best so far F0.5 {best[0]:.4f} {best[1]}')
    rule, w, t = best[1]
    md = models['dec']
    base_f = md['f_expected'] if md['rule'] == 'expected_f' else md['f_threshold']
    dec = {'rule': rule, 'w': w, 'threshold': t, 'f_val': best[0], 'f_val_gbdt_only': base_f, 'lo': LO, 'hi': HI,
           'model': MODEL_NAME}
    json.dump(dec, open(f"{cfg['out']}/validation_ce.json", 'w'), indent=1)
    K.log('CE VALIDATION macro F0.5 %.4f (GBDT only %.4f) -> w=%.2f %s' % (best[0], dec['f_val_gbdt_only'], w, rule))
    # ---------------- 3. apply to TEST ----------------------------------------------------------------
    match = {}
    for c in countries_test:
        wc = f"{cfg['work']}/test/{c}"
        if not os.path.exists(f'{wc}/pred.pkl'):
            continue
        T = pd.read_pickle(f'{wc}/pred.pkl')
        p = T.p.values.copy()
        if w < 1.0:
            h = np.flatnonzero((p >= LO) & (p <= HI))
            f_ce = f'{wc}/pce.npy'
            if os.path.exists(f_ce):
                pc = np.load(f_ce)
            else:
                T1, T2 = _texts(wc, 'E1'), _texts(wc, 'E2')
                pc = sc.predict(T1[T.i1.values[h]], T2[T.i2.values[h]]); np.save(f_ce, pc)
            p[h] = w * p[h] + (1 - w) * pc
            K.log('test', c, f'CE re-scored {len(h):,} ambiguous pairs')
        D = T[['i1', 'i2']].copy(); D['p'] = p
        D = by_expected_f(exclusive(D)) if rule == 'expected_f' else by_threshold(exclusive(D), t)
        ids1 = np.load(f'{wc}/ids1.npy', allow_pickle=True); ids2 = np.load(f'{wc}/ids2.npy', allow_pickle=True)
        g = pd.DataFrame({'s1': ids1[D.i1.values], 'c': ids2[D.i2.values]})
        match.update(g.groupby('s1', sort=False).c.agg(','.join).to_dict())
    order = K.read_tsv(f"{cfg['data']}/test/test_source1.tsv", usecols=['entity_id']).entity_id.to_numpy(dtype=object)
    out = pd.DataFrame({'source1_entity_id': order})
    out['matched_entity_ids'] = out.source1_entity_id.map(match).fillna('')
    od = cfg['out'] + '_ce'; os.makedirs(od, exist_ok=True)
    out.to_csv(f'{od}/matching_results.tsv', sep='\t', index=False)
    import shutil
    shutil.copy(f"{cfg['out']}/candidate_pairs.tsv", f'{od}/candidate_pairs.tsv')   # same candidate set
    K.log('wrote', f'{od}/matching_results.tsv', f'{len(out):,} rows')
    return dec


In [ ]:
# ---- Data: download / extract student_resource.zip and VERIFY every file -------------------------
DATA_URL = 'https://cdn.unstop.com/files/6ab10eb3b23ba_student_resource.zip'   # link from the challenge page
S3_URI = None          # or drop student_resource.zip into ROOT yourself; it is used as-is
import glob, zipfile, urllib.request
EXPECTED_ROWS = {'train_source1.tsv': 2206821, 'train_source2.tsv': 5034616, 'train_source3.tsv': 5285603,
                 'train_ground_truth.tsv': 2206821, 'test_source1.tsv': 1732544, 'test_source2.tsv': 4887273,
                 'test_source3.tsv': 5082316}          # data rows (header excluded), counted on the official files
z = f'{ROOT}/student_resource.zip'
def _find():
    hits = glob.glob(f'{ROOT}/**/train/train_source1.tsv', recursive=True)
    return os.path.dirname(os.path.dirname(hits[0])) if hits else None
def _rows(path):
    n = 0
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''): n += block.count(b'\n')
    return n - 1
def _bad_files(data):
    bad = []
    for name, exp in EXPECTED_ROWS.items():
        f = glob.glob(f'{data}/*/{name}')
        if not f: bad.append((name, 'missing')); continue
        with open(f[0], 'rb') as fh:
            fh.seek(max(0, os.path.getsize(f[0]) - 4096)); tail = fh.read()
        try: tail[tail.find(b'\n') + 1:].decode('utf-8')
        except UnicodeDecodeError: bad.append((name, 'truncated (ends mid-character)')); continue
        n = _rows(f[0])
        if n != exp: bad.append((name, f'{n:,} rows, expected {exp:,}'))
    return bad
need = 12 * 2**30
if _find() is None or _bad_files(_find()):
    if _find(): print('Damaged data found, re-extracting:', _bad_files(_find()))
    if shutil.disk_usage(ROOT).free < need: raise SystemExit('Not enough free disk to extract (need ~12 GB): free some space first')
    if not os.path.exists(z) or not zipfile.is_zipfile(z):
        if S3_URI: subprocess.run(['aws', 's3', 'cp', S3_URI, z], check=True)
        else: urllib.request.urlretrieve(DATA_URL, z + '.part'); os.replace(z + '.part', z)
    with zipfile.ZipFile(z) as zf:
        bad_member = zf.testzip()
        if bad_member: raise SystemExit(f'zip is corrupt at {bad_member}: delete {z} and re-run to download again')
        zf.extractall(f'{ROOT}/data')
DATA = _find()
problems = _bad_files(DATA)
if problems: raise SystemExit(f'Data files still damaged: {problems}. Delete {ROOT}/data and {z}, then re-run.')
print('DATA =', DATA, '| all 7 files verified (row counts + clean UTF-8 ending) | free disk %.0f GB' % (shutil.disk_usage(ROOT).free / 1e9))

In [ ]:
# ---- Config (auto-tuned to the machine) -------------------------------------------------------
FAST = False          # True: 35% of train S1; False: all training entities
SEARCH = 'exact'      # 'exact' (FAISS flat) or 'ivf' (approximate, faster)

sys.path.insert(0, SRC)
# Crash switches (only if a GPU library crashes the kernel on this machine; re-run the Config cell after changing):
os.environ['ER_GPU_PROJECT'] = '1'   # '0' -> SVD projection on CPU instead of torch GPU
os.environ['ER_NO_FAISS'] = '0'      # '1' -> skip FAISS, use torch GPU top-K
import importlib, er_fast as K; importlib.reload(K)
ram_gb = int([l for l in open('/proc/meminfo') if l.startswith('MemTotal')][0].split()[1]) / 1e6
try:
    import torch
    torch.backends.cuda.matmul.allow_tf32 = True
    gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
except Exception:
    gpus = []
print('GPUs:', gpus, '| RAM %.0f GB' % ram_gb, '| CPUs', os.cpu_count())
cfg = dict(data=DATA, work=f"{ROOT}/er_work_{'fast' if FAST else 'full'}", out=f'{ROOT}/output',
           train_frac=0.35 if FAST else 1.0, dim=192, search=SEARCH,
           ks={'name': 30, 'addr': 20, 'both': 30}, key_k=30, rev_k=3, hop_k=3, ce_train_pairs=3_000_000,
           max_train_pairs=int(min(80e6, max(20e6, ram_gb * 0.25e6))))   # ~264 B per row, keep < ~25% of RAM
RESET = False         # True once = wipe all cached stages (after data problems), then set back to False
if RESET: shutil.rmtree(cfg['work'], ignore_errors=True)
os.makedirs(cfg['work'], exist_ok=True); os.makedirs(cfg['out'], exist_ok=True)
SRC_DIR = SRC
print(cfg)

In [ ]:
# ---- Stage runner: every stage runs in its own process ------------------------------------------
# A native crash or an out-of-memory kill then ends only that process: the kernel survives, the exit
# reason is printed, and re-running the cell resumes from the stage's checkpoints.
import json, subprocess, signal
CFG_JSON = f"{cfg['work']}/cfg.json"; json.dump(cfg, open(CFG_JSON, 'w'))
LOG = f"{cfg['work']}/pipeline.log"
def run_stage(name):
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    p = subprocess.Popen([sys.executable, '-X', 'faulthandler', f"{SRC_DIR}/run_stage.py", CFG_JSON, name],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    tail = []
    with open(LOG, 'a') as lf:
        for line in p.stdout:
            if 'Warning' in line or 'warnings.warn' in line: continue
            print(line, end=''); lf.write(line); tail = (tail + [line])[-40:]
    rc = p.wait()
    if rc == 0:
        print(f'== {name}: OK'); return
    why = {-signal.SIGKILL: 'KILLED by the OS -> out of RAM. Look at the last [rss ... free ...] values above.',
           -signal.SIGSEGV: 'SEGFAULT in a native library (see the faulthandler traceback above). '
                            'If it is in torch/embedding: set ER_GPU_PROJECT=0; in faiss: set ER_NO_FAISS=1.',
           -signal.SIGABRT: 'ABORT in a native library (CUDA/FAISS). Try ER_NO_FAISS=1 or ER_GPU_PROJECT=0.'}.get(
           rc, f'exit code {rc} (Python error above)')
    raise RuntimeError(f'stage {name} failed: {why}  -- re-run this cell to resume from checkpoints')

In [ ]:
%%time
run_stage('normalise')        # Stage 1 (checkpoint per 400k-row chunk)

In [ ]:
%%time
run_stage('block')            # Stage 2 (checkpoint per view)

In [ ]:
%%time
run_stage('features_train')   # Stage 3a (checkpoint per feature)

In [ ]:
%%time
run_stage('train')            # Stage 4 (checkpoint per model); validation score -> output/validation.json
print(open(f"{cfg['out']}/validation.json").read()[:300])

In [ ]:
# (optional) free disk once models exist: train features are not needed any more
import glob
if os.path.exists(f"{cfg['work']}/models/final.pkl"):
    for d in glob.glob(f"{cfg['work']}/train/*/feats"): shutil.rmtree(d, ignore_errors=True)
print('free %.0f GB' % (shutil.disk_usage(cfg['work']).free / 1e9))

In [ ]:
%%time
run_stage('predict')          # Stage 3b + 5: test features, scoring, TSVs in output/

In [ ]:
%%time
# ---- Stage 6 (NEW): cross-encoder re-ranker (multilingual-e5-small, MIT, 118M params) ------------
# Fine-tunes on ambiguous TRAIN pairs, re-scores only pairs the GBDT is unsure about, tunes the blend
# weight on the validation entities. Writes output_ce/. If it does not help on validation, w=1 (= output/).
run_stage('ce')
print(open(f"{cfg['out']}/validation_ce.json").read())

In [ ]:
# ---- Pick the better submission by VALIDATION score ------------------------------------------
v0 = json.load(open(f"{cfg['out']}/validation.json"))
f0 = max(v0.get('f_expected', 0), v0.get('f_threshold', 0))
f1 = json.load(open(f"{cfg['out']}/validation_ce.json"))['f_val'] if os.path.exists(f"{cfg['out']}/validation_ce.json") else 0
BEST = cfg['out'] + '_ce' if f1 > f0 + 0.0005 else cfg['out']
print(f'GBDT val {f0:.4f} | +CE val {f1:.4f} -> SUBMIT {BEST}/matching_results.tsv + candidate_pairs.tsv')
d = json.load(open(f"{cfg['out']}/validation.json")).get('diagnostics')
if d: print(json.dumps(d, indent=1)[:3000])

In [ ]:
# ---- Stage 7: validate -------------------------------------------------------------------------
import glob, pandas as pd
v = glob.glob(f'{ROOT}/**/validate_submission.py', recursive=True)
if v:
    !python3 {v[0]} --matching {BEST}/matching_results.tsv --candidate {BEST}/candidate_pairs.tsv --test-dir {DATA}/test
else:
    m = pd.read_csv(f"{BEST}/matching_results.tsv", sep='\t', dtype=str, keep_default_na=False)
    s1 = pd.read_csv(f"{DATA}/test/test_source1.tsv", sep='\t', dtype=str, keep_default_na=False, usecols=['entity_id'])
    assert list(m.columns) == ['source1_entity_id', 'matched_entity_ids']
    assert m.source1_entity_id.is_unique and set(m.source1_entity_id) == set(s1.entity_id)
    assert all(len(x.split(',')) == len(set(x.split(','))) for x in m.matched_entity_ids if x)
    print('basic checks PASS', len(m), 'rows')
print(open(f"{cfg['out']}/validation.json").read()[:300])